In [1]:
# ============================================================
# D7 — Stage 4 Validation — Branch C: Deterministic Normalisation
# 0. Imports
# ============================================================

from google.colab import files
from pathlib import Path
from datetime import datetime
from difflib import SequenceMatcher

import hashlib
import json
import math
import platform
import re
import sys
import unicodedata

import pandas as pd
from scipy.optimize import linear_sum_assignment


In [2]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D7"

DOCUMENT_NAME = (
    "UK National Audit Office — Delivering STEM "
    "(science, technology, engineering and mathematics) "
    "skills for the economy"
)

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
INPUT_REPRESENTATION = "Complete deterministically normalised structural Markdown"

EXPECTED_REFERENCE_RECORD_COUNT = 59

EXPECTED_CATEGORY_COUNTS = {
    "Key fact": 10,
    "Policy context": 3,
    "Policy finding": 7,
    "Education pipeline statistic": 24,
    "Government initiative": 9,
    "Recommendation": 6
}

FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

PRIMARY_CORRECTNESS_FIELDS = FIELDS.copy()

MANDATORY_STRING_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Source Location"
]

NULLABLE_STRING_FIELDS = [
    "Unit",
    "Qualifier",
    "Reporting Period"
]

NUMERIC_FIELDS = ["Value"]

# Frozen from final D7 Branch A validation.
BLOCK_FIELDS = [
    "Category",
    "Source Location"
]

MATCH_SCORE_THRESHOLD = 0.30

MATCHING_WEIGHTS = {
    "metric": 0.50,
    "topic": 0.30,
    "statement_or_section": 0.15,
    "reporting_period": 0.05
}

EXPECTED_SOURCE_SHA256 = (
    "00cd2555312b220d7b4289144261aaba"
    "888336fb21b32b9ff53e96427a5f7eba"
)

OUTPUT_DIR = Path("outputs_D7_validation_C_revised")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected reference records:", EXPECTED_REFERENCE_RECORD_COUNT)
print("Primary correctness fields:", len(PRIMARY_CORRECTNESS_FIELDS))
print("Output directory:", OUTPUT_DIR)

Document: D7
Branch: C
Expected reference records: 59
Primary correctness fields: 9
Output directory: outputs_D7_validation_C_revised


In [31]:
# ============================================================
# 2. Upload canonical Branch C validation inputs
# ============================================================
# Required:
#   1) D7_reference_values.csv
#   2) D7_branch_C_parsed_extraction.json
#   3) D7_branch_C_structure_check.json
#   4) D7_branch_C_experiment_metadata.json
#   5) D7_branch_C_normalisation_check.json
#
# Optional:
#   6) D7_branch_C_experiment_summary.json
#
# IMPORTANT:
# The structure-check artefact and experiment-summary artefact can
# contain overlapping diagnostic keys. Therefore, file identification
# is based first on canonical filename patterns and only secondarily
# on content-based checks.

print(
    "Upload:\n"
    "1. D7_reference_values.csv\n"
    "2. D7_branch_C_parsed_extraction.json\n"
    "3. D7_branch_C_structure_check.json\n"
    "4. D7_branch_C_experiment_metadata.json\n"
    "5. D7_branch_C_normalisation_check.json\n"
    "6. Optional: D7_branch_C_experiment_summary.json"
)

uploaded = files.upload()

uploaded_paths = [
    Path(name)
    for name in uploaded
]


# ------------------------------------------------------------
# Separate CSV and JSON files
# ------------------------------------------------------------

csv_paths = [
    path
    for path in uploaded_paths
    if path.suffix.lower() == ".csv"
]

json_paths = [
    path
    for path in uploaded_paths
    if path.suffix.lower() == ".json"
]


if len(csv_paths) != 1:
    raise ValueError(
        "Upload exactly one CSV file: D7_reference_values.csv."
    )


REFERENCE_PATH = csv_paths[0]


# ------------------------------------------------------------
# Initialise artefact paths
# ------------------------------------------------------------

EXTRACTION_PATH = None

STRUCTURE_CHECK_PATH = None

EXPERIMENT_METADATA_PATH = None

NORMALISATION_INTEGRITY_PATH = None

EXPERIMENT_SUMMARY_PATH = None


# ------------------------------------------------------------
# Helper for filename matching
# ------------------------------------------------------------

def canonical_filename(path):

    return (
        path.name
        .casefold()
        .replace(" ", "_")
    )


# ------------------------------------------------------------
# FIRST PASS:
# identify artefacts from canonical filename patterns
# ------------------------------------------------------------

for path in json_paths:

    filename = canonical_filename(
        path
    )

    if (
        "d7_branch_c_parsed_extraction"
        in filename
    ):

        EXTRACTION_PATH = path

        continue


    if (
        "d7_branch_c_structure_check"
        in filename
    ):

        STRUCTURE_CHECK_PATH = path

        continue


    if (
        "d7_branch_c_experiment_metadata"
        in filename
    ):

        EXPERIMENT_METADATA_PATH = path

        continue


    if (
        "d7_branch_c_normalisation_check"
        in filename
        or
        "d7_branch_c_normalization_check"
        in filename
    ):

        NORMALISATION_INTEGRITY_PATH = path

        continue


    if (
        "d7_branch_c_experiment_summary"
        in filename
    ):

        EXPERIMENT_SUMMARY_PATH = path

        continue


# ------------------------------------------------------------
# SECOND PASS:
# content-based fallback only for artefacts not yet identified
# ------------------------------------------------------------

for path in json_paths:

    with path.open(
        "r",
        encoding="utf-8-sig"
    ) as file:

        obj = json.load(
            file
        )


    if not isinstance(
        obj,
        dict
    ):
        continue


    # --------------------------------------------------------
    # Parsed extraction
    # --------------------------------------------------------

    if (
        EXTRACTION_PATH is None
        and obj.get(
            "document_id"
        ) == DOCUMENT_ID
        and obj.get(
            "branch"
        ) == BRANCH
        and isinstance(
            obj.get(
                "records"
            ),
            list
        )
    ):

        EXTRACTION_PATH = path

        continue


    # --------------------------------------------------------
    # Experiment metadata
    # --------------------------------------------------------

    if (
        EXPERIMENT_METADATA_PATH is None
        and obj.get(
            "document_id"
        ) == DOCUMENT_ID
        and obj.get(
            "branch"
        ) == BRANCH
        and "parsed_extraction_sha256"
        in obj
        and "source_sha256"
        in obj
        and "structure_check_file"
        in obj
    ):

        EXPERIMENT_METADATA_PATH = path

        continue


    # --------------------------------------------------------
    # Normalisation check
    # --------------------------------------------------------

    if (
        NORMALISATION_INTEGRITY_PATH is None
        and obj.get(
            "document_id"
        ) == DOCUMENT_ID
        and obj.get(
            "branch"
        ) == BRANCH
        and obj.get(
            "parent_branch"
        ) == "B"
        and "normalisation_integrity_passed"
        in obj
        and "parent_equivalence_passed"
        in obj
    ):

        NORMALISATION_INTEGRITY_PATH = path

        continue


    # --------------------------------------------------------
    # Experiment summary
    # --------------------------------------------------------
    #
    # Explicitly identify this BEFORE the structure-check
    # fallback because summary files may also contain structure
    # diagnostics.
    # --------------------------------------------------------

    if (
        EXPERIMENT_SUMMARY_PATH is None
        and obj.get(
            "document_id"
        ) == DOCUMENT_ID
        and obj.get(
            "branch"
        ) == BRANCH
        and "validation_status"
        in obj
        and "accuracy_validation_completed"
        in obj
    ):

        EXPERIMENT_SUMMARY_PATH = path

        continue


    # --------------------------------------------------------
    # Structure check
    # --------------------------------------------------------
    #
    # A valid structure-check candidate:
    #   - belongs to D7 / Branch C
    #   - contains structure/schema diagnostics
    #   - is NOT an experiment summary
    #   - is NOT experiment metadata
    # --------------------------------------------------------

    if (
        STRUCTURE_CHECK_PATH is None
        and obj.get(
            "document_id"
        ) == DOCUMENT_ID
        and obj.get(
            "branch"
        ) == BRANCH
        and "structure_valid"
        in obj
        and "record_schema_valid"
        in obj
        and "field_types_valid"
        in obj
        and "validation_status"
        not in obj
        and "accuracy_validation_completed"
        not in obj
        and "parsed_extraction_sha256"
        not in obj
    ):

        STRUCTURE_CHECK_PATH = path

        continue


# ------------------------------------------------------------
# Final required-file checks
# ------------------------------------------------------------

if EXTRACTION_PATH is None:

    raise ValueError(
        "Could not identify D7_branch_C_parsed_extraction.json."
    )


if STRUCTURE_CHECK_PATH is None:

    raise ValueError(
        "Could not identify D7_branch_C_structure_check.json."
    )


if EXPERIMENT_METADATA_PATH is None:

    raise ValueError(
        "Could not identify D7_branch_C_experiment_metadata.json."
    )


if NORMALISATION_INTEGRITY_PATH is None:

    raise ValueError(
        "Could not identify D7_branch_C_normalisation_check.json."
    )


# ------------------------------------------------------------
# Verify that files are not accidentally assigned twice
# ------------------------------------------------------------

required_paths = {
    "parsed_extraction":
        EXTRACTION_PATH,

    "structure_check":
        STRUCTURE_CHECK_PATH,

    "experiment_metadata":
        EXPERIMENT_METADATA_PATH,

    "normalisation_check":
        NORMALISATION_INTEGRITY_PATH
}


required_path_strings = [
    str(path)
    for path
    in required_paths.values()
]


if (
    len(
        required_path_strings
    )
    !=
    len(
        set(
            required_path_strings
        )
    )
):

    raise ValueError(
        "The same JSON file was assigned to more than one "
        "required Branch C artefact type."
    )


# ------------------------------------------------------------
# Display identified artefacts
# ------------------------------------------------------------

print(
    "\nIdentified D7 Branch C validation inputs:"
)

print(
    "Reference:",
    REFERENCE_PATH.name
)

print(
    "Parsed extraction:",
    EXTRACTION_PATH.name
)

print(
    "Structure check:",
    STRUCTURE_CHECK_PATH.name
)

print(
    "Experiment metadata:",
    EXPERIMENT_METADATA_PATH.name
)

print(
    "Normalisation integrity:",
    NORMALISATION_INTEGRITY_PATH.name
)

print(
    "Experiment summary:",
    (
        EXPERIMENT_SUMMARY_PATH.name
        if EXPERIMENT_SUMMARY_PATH
        is not None
        else "Not supplied"
    )
)

Upload:
1. D7_reference_values.csv
2. D7_branch_C_parsed_extraction.json
3. D7_branch_C_structure_check.json
4. D7_branch_C_experiment_metadata.json
5. D7_branch_C_normalisation_check.json
6. Optional: D7_branch_C_experiment_summary.json


Saving D7_branch_C_structure_check.json to D7_branch_C_structure_check (1).json
Saving D7_branch_C_parsed_extraction.json to D7_branch_C_parsed_extraction (1).json
Saving D7_branch_C_normalisation_check.json to D7_branch_C_normalisation_check (1).json
Saving D7_branch_C_experiment_summary.json to D7_branch_C_experiment_summary (1).json
Saving D7_branch_C_experiment_metadata.json to D7_branch_C_experiment_metadata (1).json
Saving D7_reference_values.csv to D7_reference_values (1).csv

Identified D7 Branch C validation inputs:
Reference: D7_reference_values (1).csv
Parsed extraction: D7_branch_C_parsed_extraction (1).json
Structure check: D7_branch_C_structure_check (1).json
Experiment metadata: D7_branch_C_experiment_metadata (1).json
Normalisation integrity: D7_branch_C_normalisation_check (1).json
Experiment summary: D7_branch_C_experiment_summary (1).json


In [32]:
# ============================================================
# 3. File hashing utility and input hashes
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open("rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


REFERENCE_SHA256 = sha256_file(REFERENCE_PATH)
EXTRACTION_SHA256 = sha256_file(EXTRACTION_PATH)
STRUCTURE_CHECK_SHA256 = sha256_file(STRUCTURE_CHECK_PATH)
EXPERIMENT_METADATA_SHA256 = sha256_file(EXPERIMENT_METADATA_PATH)
NORMALISATION_INTEGRITY_SHA256 = sha256_file(NORMALISATION_INTEGRITY_PATH)

EXPERIMENT_SUMMARY_SHA256 = (
    sha256_file(EXPERIMENT_SUMMARY_PATH)
    if EXPERIMENT_SUMMARY_PATH is not None
    else None
)

print("Reference SHA-256:", REFERENCE_SHA256)
print("Extraction SHA-256:", EXTRACTION_SHA256)
print("Structure-check SHA-256:", STRUCTURE_CHECK_SHA256)
print("Experiment-metadata SHA-256:", EXPERIMENT_METADATA_SHA256)
print("Normalisation-integrity SHA-256:", NORMALISATION_INTEGRITY_SHA256)

Reference SHA-256: c89fea594b5419b32a59cac414137ebde23e3c72c9e44db88280eb4d21edcf88
Extraction SHA-256: d703661058b0bab70d290db40f17606f0e7b0717afa876dfdf06ad777889e274
Structure-check SHA-256: e790b1d657092a671b0d9149f3b42c3d47c5237f6cbf32d41e8018d4857ef2be
Experiment-metadata SHA-256: 5bbbb25e32caba4f2b18cc9c2e9e49c7a4467d6e37d5ae77b875c36cf2b739ca
Normalisation-integrity SHA-256: ac1d4af1e9bc67223665135f0119fe4764bcb9792747eb45446c0dd0da592e16


In [33]:
# ============================================================
# 4. Load fixed Stage 1 reference dataset
# ============================================================

reference_df = pd.read_csv(
    REFERENCE_PATH,
    dtype=object,
    keep_default_na=False,
    encoding="utf-8-sig"
)


def restore_csv_null(value):
    if value is None:
        return None

    if isinstance(value, str) and value == "":
        return None

    return value


def restore_mixed_number(value):
    if value is None or not isinstance(value, str):
        return value

    text = (
        value.strip()
        .replace(",", "")
        .replace("−", "-")
        .replace("–", "-")
    )

    if re.fullmatch(r"-?\d+", text):
        return int(text)

    if re.fullmatch(r"-?\d+\.\d+", text):
        return float(text)

    return value


for column in reference_df.columns:
    reference_df[column] = reference_df[column].map(
        restore_csv_null
    )

reference_df["Value"] = reference_df["Value"].map(
    restore_mixed_number
)

print("Reference shape:", reference_df.shape)
print("Reference columns:", reference_df.columns.tolist())

display(reference_df.head(10))

Reference shape: (59, 9)
Reference columns: ['Category', 'Statement or Section', 'Metric', 'Topic', 'Value', 'Unit', 'Qualifier', 'Reporting Period', 'Source Location']


,Category,Statement or Section,Metric,Topic,Value,Unit,Qualifier,Reporting Period,Source Location
0,Key fact,Key facts,"Spent on, or committed to, key STEM-specific i...",STEM interventions,990.0,GBP million,None,2007 to autumn 2017,PDF page 6 — Key facts
1,Key fact,Key facts,Undergraduate enrolments in STEM subjects,Undergraduate STEM participation,442000.0,enrolments,None,2015/16,PDF page 6 — Key facts
2,Key fact,Key facts,Graduates in STEM subjects known to be working...,STEM graduate destinations,24.0,percent,None,Six months after graduation,PDF page 6 — Key facts
3,Key fact,Key facts,Additional STEM technicians estimated as neede...,STEM technician demand,700000.0,technicians,None,Decade to 2024,PDF page 6 — Key facts
4,Key fact,Key facts,STEM apprenticeship starts,STEM apprenticeships,112000.0,starts,None,2016/17,PDF page 6 — Key facts
5,Key fact,Key facts,STEM apprenticeships started by women,Female participation in STEM apprenticeships,8.0,percent,None,2016/17,PDF page 6 — Key facts
6,Key fact,Key facts,Government investment in national colleges,National colleges,80.0,GBP million,None,None,PDF page 6 — Key facts
7,Key fact,Key facts,Rise in STEM A level examination entries compa...,STEM A level entries,2.6,percent,None,2016/17,PDF page 6 — Key facts
8,Key fact,Key facts,Fall in enrolments in part-time undergraduate ...,Part-time undergraduate STEM participation,-30.9,percent,None,2011/12 to 2015/16,PDF page 6 — Key facts
9,Key fact,Key facts,Government capital investment in higher educat...,Higher education STEM provision,200.0,GBP million,None,2015/16,PDF page 6 — Key facts


In [34]:
# ============================================================
# 5. Load canonical preserved Branch C extraction
# ============================================================

with EXTRACTION_PATH.open(
    "r",
    encoding="utf-8"
) as file:
    extraction_content = json.load(file)

valid_json = True

top_level_object_valid = isinstance(
    extraction_content,
    dict
)

document_id_correct = (
    top_level_object_valid
    and extraction_content.get("document_id") == DOCUMENT_ID
)

branch_correct = (
    top_level_object_valid
    and extraction_content.get("branch") == BRANCH
)

records_is_list = (
    top_level_object_valid
    and isinstance(
        extraction_content.get("records"),
        list
    )
)

if not records_is_list:
    raise ValueError(
        "Use the canonical D7_branch_C_parsed_extraction.json "
        "generated by the current Branch C notebook."
    )

extracted_records = extraction_content["records"]

print("Document ID correct:", document_id_correct)
print("Branch correct:", branch_correct)
print("Extracted records:", len(extracted_records))

Document ID correct: True
Branch correct: True
Extracted records: 60


In [35]:
# ============================================================
# 6. Load Branch C structure check and experiment metadata
# ============================================================

with STRUCTURE_CHECK_PATH.open(
    "r",
    encoding="utf-8"
) as file:
    branch_c_structure_check = json.load(file)

with EXPERIMENT_METADATA_PATH.open(
    "r",
    encoding="utf-8"
) as file:
    branch_c_experiment_metadata = json.load(file)

branch_c_structure_valid = bool(
    branch_c_structure_check.get("structure_valid")
)

metadata_document_id_correct = (
    branch_c_experiment_metadata.get("document_id")
    == DOCUMENT_ID
)

metadata_branch_correct = (
    branch_c_experiment_metadata.get("branch")
    == BRANCH
)

metadata_source_hash = (
    branch_c_experiment_metadata.get("source_sha256")
)

metadata_source_hash_correct = (
    metadata_source_hash
    == EXPECTED_SOURCE_SHA256
)

metadata_extraction_hash = (
    branch_c_experiment_metadata.get(
        "parsed_extraction_sha256"
    )
)

parsed_extraction_hash_matches_metadata = (
    metadata_extraction_hash
    == EXTRACTION_SHA256
)

print("Branch C structure valid:", branch_c_structure_valid)
print("Metadata document ID correct:", metadata_document_id_correct)
print("Metadata branch correct:", metadata_branch_correct)
print("Source hash matches Stage 1:", metadata_source_hash_correct)
print(
    "Parsed extraction hash matches Branch C metadata:",
    parsed_extraction_hash_matches_metadata
)

if not parsed_extraction_hash_matches_metadata:
    raise AssertionError(
        "The uploaded parsed extraction is not the exact file "
        "recorded by the Branch C experiment metadata."
    )

if not metadata_source_hash_correct:
    raise AssertionError(
        "The Branch C source hash does not match the D7 Stage 1 source."
    )

Branch C structure valid: True
Metadata document ID correct: True
Metadata branch correct: True
Source hash matches Stage 1: True
Parsed extraction hash matches Branch C metadata: True


In [36]:
# ============================================================
# 7. Preserve observed records and assess record schema
# ============================================================

comparison_record_rows = []
schema_issue_rows = []
field_order_issue_rows = []

for record_index, record in enumerate(extracted_records):

    if not isinstance(record, dict):
        schema_issue_rows.append(
            {
                "Record Index": record_index,
                "Issue": "Record is not a JSON object"
            }
        )

        comparison_record = {
            field: None
            for field in FIELDS
        }

    else:
        observed_fields = list(record.keys())

        missing_fields = [
            field for field in FIELDS
            if field not in record
        ]

        extra_fields = [
            field for field in observed_fields
            if field not in FIELDS
        ]

        if missing_fields or extra_fields:
            schema_issue_rows.append(
                {
                    "Record Index": record_index,
                    "Issue": "Missing or unexpected field names",
                    "Missing Fields": missing_fields,
                    "Extra Fields": extra_fields,
                    "Expected Fields": FIELDS,
                    "Observed Fields": observed_fields
                }
            )

        # JSON object order is recorded diagnostically but is not
        # treated as semantic schema invalidity.
        if (
            not missing_fields
            and not extra_fields
            and observed_fields != FIELDS
        ):
            field_order_issue_rows.append(
                {
                    "Record Index": record_index,
                    "Expected Fields": FIELDS,
                    "Observed Fields": observed_fields
                }
            )

        # Missing fields remain missing in the raw extraction.
        # The comparison copy receives None only to permit
        # deterministic downstream evaluation; no raw repair occurs.
        comparison_record = {
            field: record.get(field)
            for field in FIELDS
        }

    comparison_record["_extraction_index"] = record_index
    comparison_record_rows.append(comparison_record)


extracted_df = pd.DataFrame(comparison_record_rows)

schema_issues_df = pd.DataFrame(schema_issue_rows)
field_order_issues_df = pd.DataFrame(field_order_issue_rows)

record_schema_valid = schema_issues_df.empty

print("Record schema valid:", record_schema_valid)
print("Schema issue count:", len(schema_issues_df))
print("Field-order diagnostics:", len(field_order_issues_df))

if not schema_issues_df.empty:
    display(schema_issues_df)

display(extracted_df.head(10))

Record schema valid: True
Schema issue count: 0
Field-order diagnostics: 0


,Category,Statement or Section,Metric,Topic,Value,Unit,Qualifier,Reporting Period,Source Location,_extraction_index
0,Key fact,Key facts,Spending or commitments on key STEM-specific i...,Key STEM-specific interventions,990.0,£ million,None,between 2007 and autumn 2017,PDF page 6 — Key facts,0
1,Key fact,Key facts,Undergraduate enrolments in STEM subjects,Undergraduate STEM subjects,442000.0,enrolments,None,2015/16,PDF page 6 — Key facts,1
2,Key fact,Key facts,Graduates in STEM subjects known to be working...,STEM graduates,24.0,%,None,six months later,PDF page 6 — Key facts,2
3,Key fact,Key facts,Additional STEM technicians estimated to be ne...,STEM technicians,700000.0,technicians,None,decade to 2024,PDF page 6 — Key facts,3
4,Key fact,Key facts,STEM apprenticeship starts,STEM apprenticeships,112000.0,starts,None,2016/17,PDF page 6 — Key facts,4
5,Key fact,Key facts,STEM apprenticeships started by women,Women in STEM apprenticeships,8.0,%,None,2016/17,PDF page 6 — Key facts,5
6,Key fact,Key facts,Government investment in national colleges,National colleges,80.0,£ million,None,None,PDF page 6 — Key facts,6
7,Key fact,Key facts,Rise in STEM A level examination entries,STEM A level examination entries,2.6,%,None,2016/17 compared with the previous year,PDF page 6 — Key facts,7
8,Key fact,Key facts,Fall in enrolments in part-time undergraduate ...,Part-time undergraduate STEM degrees,-30.9,%,None,between 2011/12 and 2015/16,PDF page 6 — Key facts,8
9,Key fact,Key facts,Government capital investment in higher educat...,Higher education STEM provision,200.0,£ million,None,2015/16,PDF page 6 — Key facts,9


In [37]:
# ============================================================
# 8. Reference integrity and extraction content diagnostics
# ============================================================

reference_schema_valid = (
    reference_df.columns.tolist()
    == FIELDS
)

reference_record_count_valid = (
    len(reference_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"]
    .value_counts()
    .to_dict()
)

reference_category_counts_valid = (
    reference_category_counts
    == EXPECTED_CATEGORY_COUNTS
)

extraction_record_count_valid = (
    len(extracted_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)

extraction_category_counts = (
    extracted_df["Category"]
    .value_counts(dropna=False)
    .to_dict()
)

extraction_category_counts_valid = (
    extraction_category_counts
    == EXPECTED_CATEGORY_COUNTS
)

print("Reference schema valid:", reference_schema_valid)
print("Reference count valid:", reference_record_count_valid)
print("Reference category counts valid:", reference_category_counts_valid)
print("Extraction count matches reference:", extraction_record_count_valid)
print("Extraction category counts match reference:", extraction_category_counts_valid)

if not reference_schema_valid:
    raise AssertionError(
        "The fixed D7 Stage 1 reference schema is invalid."
    )

if not reference_record_count_valid:
    raise AssertionError(
        "The fixed D7 Stage 1 reference record count is invalid."
    )

if not reference_category_counts_valid:
    raise AssertionError(
        "The fixed D7 Stage 1 reference category counts are invalid."
    )

Reference schema valid: True
Reference count valid: True
Reference category counts valid: True
Extraction count matches reference: False
Extraction category counts match reference: False


In [38]:
# ============================================================
# 9. Null-safe type and mandatory-content diagnostics
# ============================================================

def is_null(value):
    if value is None:
        return True

    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


type_issue_rows = []
missing_mandatory_rows = []

for row_index, row in extracted_df.iterrows():

    for field in MANDATORY_STRING_FIELDS:
        value = row[field]

        if is_null(value) or value == "":
            missing_mandatory_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field
                }
            )

        elif not isinstance(value, str):
            type_issue_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field,
                    "Observed Type": type(value).__name__,
                    "Expected Type": "string"
                }
            )

    for field in NULLABLE_STRING_FIELDS:
        value = row[field]

        if (
            not is_null(value)
            and not isinstance(value, str)
        ):
            type_issue_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field,
                    "Observed Type": type(value).__name__,
                    "Expected Type": "string or null"
                }
            )

    value = row["Value"]

    if (
        not is_null(value)
        and (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
        )
    ):
        type_issue_rows.append(
            {
                "Record Index": int(row_index),
                "Field": "Value",
                "Observed Type": type(value).__name__,
                "Expected Type": "number or null"
            }
        )


type_issues_df = pd.DataFrame(type_issue_rows)
missing_mandatory_df = pd.DataFrame(missing_mandatory_rows)

field_types_valid = type_issues_df.empty
mandatory_fields_complete = missing_mandatory_df.empty

print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)
print("Type issues:", len(type_issues_df))
print("Missing mandatory values:", len(missing_mandatory_df))

if not type_issues_df.empty:
    display(type_issues_df)

if not missing_mandatory_df.empty:
    display(missing_mandatory_df)

Field types valid: True
Mandatory fields complete: True
Type issues: 0
Missing mandatory values: 0


In [39]:
# ============================================================
# 10. Comparison-only text normalisation and similarity
# ============================================================
# FROZEN FROM FINAL D7 BRANCH A VALIDATION — REUSED UNCHANGED.

def normalise_text(value):
    if is_null(value):
        return None

    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    replacements = {
        "\u00a0": " ",
        "\u2007": " ",
        "\u202f": " ",
        "\u2010": "-",
        "\u2011": "-",
        "\u2012": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2212": "-",
        "£": "gbp "
    }

    for source, target in replacements.items():
        text = text.replace(source, target)

    text = re.sub(r"\s+", " ", text)

    return text.strip().casefold()


STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "for", "from",
    "in", "is", "of", "on", "or", "the", "to", "was",
    "were", "with"
}


def comparison_tokens(value):
    text = normalise_text(value)

    if text is None:
        return set()

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text
    )

    return {
        token
        for token in text.split()
        if token and token not in STOPWORDS
    }


def sequence_similarity(first, second):
    first_text = normalise_text(first)
    second_text = normalise_text(second)

    if first_text is None and second_text is None:
        return 1.0

    if first_text is None or second_text is None:
        return 0.0

    return SequenceMatcher(
        None,
        first_text,
        second_text
    ).ratio()


def jaccard_similarity(first, second):
    first_tokens = comparison_tokens(first)
    second_tokens = comparison_tokens(second)

    if not first_tokens and not second_tokens:
        return 1.0

    if not first_tokens or not second_tokens:
        return 0.0

    return (
        len(first_tokens & second_tokens)
        / len(first_tokens | second_tokens)
    )


def text_similarity(first, second):
    return max(
        sequence_similarity(first, second),
        jaccard_similarity(first, second)
    )

In [40]:
# ============================================================
# 11. Frozen D7 source-grounded equivalence rules
# ============================================================
# FROZEN FROM FINAL D7 BRANCH A VALIDATION — REUSED UNCHANGED.

# These rules were established from the D7 source document,
# Stage 1 reference definitions and the first conservative
# Branch A validation run.
#
# They represent alternative source-grounded formulations of
# the same observation. They must now be frozen and reused
# unchanged for D7 Branches A, B and C.
#
# IMPORTANT:
# - No Value is used in these rules.
# - Unit and Qualifier remain separate fields.
# - Qualifier wording is never moved into Unit.
# - Generic fuzzy similarity is NOT used for correctness.
# - Lexical similarity remains diagnostic / alignment support only.


# ------------------------------------------------------------
# 12.1 Helper for controlled equivalence pairs
# ------------------------------------------------------------

def make_equivalence_pair(first, second):
    return frozenset(
        {
            normalise_text(first),
            normalise_text(second)
        }
    )


def pair_is_controlled_equivalent(
    first,
    second,
    equivalence_pairs
):
    first_text = normalise_text(first)
    second_text = normalise_text(second)

    if first_text == second_text:
        return True

    pair = frozenset(
        {
            first_text,
            second_text
        }
    )

    return pair in equivalence_pairs


# ------------------------------------------------------------
# 12.2 Unit equivalence
# ------------------------------------------------------------

UNIT_EQUIVALENCE_MAP = {
    "%": "percent",
    "percent": "percent",

    "£ million": "gbp million",
    "gbp million": "gbp million"
}


def canonical_unit(value):
    text = normalise_text(value)

    if text is None:
        return None

    normalised_mapping = {
        normalise_text(key): normalise_text(target)
        for key, target in UNIT_EQUIVALENCE_MAP.items()
    }

    return normalised_mapping.get(
        text,
        text
    )


CONTROLLED_UNIT_EQUIVALENCE_PAIRS = {
    # Counts of people who have graduated with STEM degrees.
    make_equivalence_pair(
        "graduates",
        "people"
    ),

    # The source refers to 15 clear routes into careers.
    make_equivalence_pair(
        "career routes",
        "routes"
    )
}


def is_unit_equivalent(
    reference_value,
    extracted_value
):
    if (
        canonical_unit(reference_value)
        == canonical_unit(extracted_value)
    ):
        return True

    return pair_is_controlled_equivalent(
        reference_value,
        extracted_value,
        CONTROLLED_UNIT_EQUIVALENCE_PAIRS
    )


# ------------------------------------------------------------
# 12.3 Reporting-period equivalence
# ------------------------------------------------------------

PERIOD_EQUIVALENCE_MAP = {
    "between 2007 and autumn 2017":
        "2007 to autumn 2017",

    "2007 to autumn 2017":
        "2007 to autumn 2017",

    "six months later":
        "six months after graduation",

    "six months after graduation":
        "six months after graduation",

    "within six months of graduation":
        "within six months",

    "within six months":
        "within six months",

    "2016/17 compared with the previous year":
        "2016/17",

    "2016/17 versus previous year":
        "2016/17",

    "between 2011/12 and 2015/16":
        "2011/12 to 2015/16",

    "2011/12 to 2015/16":
        "2011/12 to 2015/16",

    "between 2011/12 and 2016/17":
        "2011/12 to 2016/17",

    "2011/12 to 2016/17":
        "2011/12 to 2016/17"
}


def canonical_period(value):
    text = normalise_text(value)

    if text is None:
        return None

    normalised_mapping = {
        normalise_text(key): normalise_text(target)
        for key, target in PERIOD_EQUIVALENCE_MAP.items()
    }

    return normalised_mapping.get(
        text,
        text
    )


def is_period_equivalent(
    reference_value,
    extracted_value
):
    return (
        canonical_period(reference_value)
        == canonical_period(extracted_value)
    )


# ------------------------------------------------------------
# 12.4 Qualifier equivalence
# ------------------------------------------------------------

# "minimum" and "and upwards" express the same lower-bound
# condition for institutes of technology targeting level 4
# and above.
#
# Other observed qualifiers such as "only", "just" and
# "additional" are NOT treated as equivalent to null because
# the D7 Qualifier field is intended for approximation /
# inequality information rather than emphasis.

CONTROLLED_QUALIFIER_EQUIVALENCE_PAIRS = {
    make_equivalence_pair(
        "minimum",
        "and upwards"
    )
}


def canonical_qualifier(value):
    return normalise_text(value)


def is_qualifier_equivalent(
    reference_value,
    extracted_value
):
    if (
        canonical_qualifier(reference_value)
        == canonical_qualifier(extracted_value)
    ):
        return True

    return pair_is_controlled_equivalent(
        reference_value,
        extracted_value,
        CONTROLLED_QUALIFIER_EQUIVALENCE_PAIRS
    )


# ------------------------------------------------------------
# 12.5 Statement / Section equivalence
# ------------------------------------------------------------

def canonical_statement(value):
    text = normalise_text(value)

    if text is None:
        return None

    summary_match = re.search(
        r"summary paragraph\s+(\d+)",
        text
    )

    if summary_match:
        return (
            f"summary paragraph "
            f"{summary_match.group(1)}"
        )

    recommendation_match = re.search(
        r"recommendation\s+"
        r"(\d+)\s*\(?([a-f])\)?",
        text
    )

    if recommendation_match:
        return (
            "recommendation "
            f"{recommendation_match.group(1)}"
            f"({recommendation_match.group(2)})"
        )

    return text


# The Stage 1 reference generally uses a paragraph/recommendation
# identifier, whereas Branch A often uses a source-grounded
# subsection heading or concise statement label.
#
# These alternatives are tied to the physical source location,
# preventing arbitrary cross-record equivalence.

D7_ALLOWED_STATEMENT_BY_SOURCE = {
    normalise_text(
        "PDF page 7 — Summary paragraph 1"
    ): {
        normalise_text("Background")
    },

    normalise_text(
        "PDF page 7 — Summary paragraph 2"
    ): {
        normalise_text("Background")
    },

    normalise_text(
        "PDF page 7 — Summary paragraph 3"
    ): {
        normalise_text("Background")
    },

    normalise_text(
        "PDF page 7 — Summary paragraph 4"
    ): {
        normalise_text(
            "Government intervention"
        )
    },

    normalise_text(
        "PDF page 8 — Summary paragraph 5"
    ): {
        normalise_text(
            "Government intervention"
        )
    },

    normalise_text(
        "PDF page 8 — Summary paragraph 6"
    ): {
        normalise_text(
            "Scope and approach"
        )
    },

    normalise_text(
        "PDF page 8 — Summary paragraph 7"
    ): {
        normalise_text(
            "Government's understanding of the need "
            "for enhanced STEM skills in the workforce"
        )
    },

    normalise_text(
        "PDF page 9 — Summary paragraph 8"
    ): {
        normalise_text(
            "Government's understanding of the need "
            "for enhanced STEM skills in the workforce"
        )
    },

    normalise_text(
        "PDF page 9 — Summary paragraph 9"
    ): {
        normalise_text(
            "Government's understanding of the need "
            "for enhanced STEM skills in the workforce"
        )
    },

    normalise_text(
        "PDF page 9 — Summary paragraph 10"
    ): {
        normalise_text(
            "Government's understanding of the need "
            "for enhanced STEM skills in the workforce"
        )
    },

    normalise_text(
        "PDF page 9 — Summary paragraph 11"
    ): {
        normalise_text(
            "Government's understanding of the need "
            "for enhanced STEM skills in the workforce"
        )
    },

    normalise_text(
        "PDF page 9 — Summary paragraph 12"
    ): {
        normalise_text(
            "Government's understanding of the need "
            "for enhanced STEM skills in the workforce"
        )
    },

    normalise_text(
        "PDF page 10 — Summary paragraph 13"
    ): {
        normalise_text(
            "The performance of the education pipeline "
            "in delivering STEM skills"
        )
    },

    normalise_text(
        "PDF page 10 — Summary paragraph 14"
    ): {
        normalise_text(
            "Females are underrepresented in most STEM "
            "subject areas at every stage of the STEM skills pipeline"
        )
    },

    normalise_text(
        "PDF page 10 — Summary paragraph 15"
    ): {
        normalise_text(
            "The number of people participating in STEM-related "
            "vocational courses has risen in some areas but not others"
        )
    },

    normalise_text(
        "PDF page 10 — Summary paragraph 16"
    ): {
        normalise_text(
            "Enrolments in undergraduate STEM courses "
            "have fallen slightly since 2011/12"
        )
    },

    normalise_text(
        "PDF page 11 — Summary paragraph 17"
    ): {
        normalise_text(
            "Graduate outcomes from STEM degrees"
        )
    },

    normalise_text(
        "PDF page 11 — Summary paragraph 18"
    ): {
        normalise_text(
            "The latest initiatives designed to enhance "
            "the development of STEM skills"
        )
    },

    normalise_text(
        "PDF page 11 — Summary paragraph 19"
    ): {
        normalise_text(
            "The latest initiatives designed to enhance "
            "the development of STEM skills"
        )
    },

    normalise_text(
        "PDF page 11 — Summary paragraph 20"
    ): {
        normalise_text(
            "Schools sector teacher supply initiatives"
        )
    },

    normalise_text(
        "PDF page 11 — Summary paragraph 21"
    ): {
        normalise_text(
            "Conclusion on value for money"
        )
    },

    normalise_text(
        "PDF page 12 — Recommendation 22(a)"
    ): {
        normalise_text("DfE should")
    },

    normalise_text(
        "PDF page 12 — Recommendation 22(b)"
    ): {
        normalise_text("DfE should")
    },

    normalise_text(
        "PDF page 12 — Recommendation 23(c)"
    ): {
        normalise_text("BEIS should")
    },

    normalise_text(
        "PDF page 12 — Recommendation 23(d)"
    ): {
        normalise_text("BEIS should")
    },

    normalise_text(
        "PDF page 12 — Recommendation 24(e)"
    ): {
        normalise_text(
            "DfE and other key departments should"
        )
    },

    normalise_text(
        "PDF page 12 — Recommendation 24(f)"
    ): {
        normalise_text(
            "DfE and other key departments should"
        )
    }
}


def is_statement_equivalent(
    reference_value,
    extracted_value,
    source_location
):
    if (
        canonical_statement(reference_value)
        == canonical_statement(extracted_value)
    ):
        return True

    source_key = normalise_text(
        source_location
    )

    extracted_text = normalise_text(
        extracted_value
    )

    allowed_values = (
        D7_ALLOWED_STATEMENT_BY_SOURCE.get(
            source_key,
            set()
        )
    )

    return extracted_text in allowed_values


# ------------------------------------------------------------
# 12.6 Metric equivalence
# ------------------------------------------------------------

CONTROLLED_METRIC_EQUIVALENCE_PAIRS = {
    make_equivalence_pair(
        "Female students as a share of all STEM A level examination entries",
        "Female share of all STEM A level exam entries"
    ),

    make_equivalence_pair(
        "Female students as a share of computing examination entries",
        "Female share of computing examination entries"
    ),

    make_equivalence_pair(
        "Female students as a share of physics examination entries",
        "Female share of physics examination entries"
    ),

    make_equivalence_pair(
        "Female students as a share of mathematics examination entries",
        "Female share of mathematics examination entries"
    ),

    make_equivalence_pair(
        "Female students as a share of STEM apprenticeship starts",
        "Female share of STEM apprenticeship starts"
    ),

    make_equivalence_pair(
        "Women as a share of all apprenticeship starts",
        "Female share of all apprenticeship starts"
    ),

    make_equivalence_pair(
        "Female students as a share of undergraduate STEM enrolments",
        "Female share of undergraduate STEM enrolments"
    ),

    make_equivalence_pair(
        "Women as a share of all undergraduate enrolments",
        "Female share of all undergraduate enrolments"
    ),

    make_equivalence_pair(
        "Non-apprenticeship STEM further education learning aims",
        "Non-apprenticeship STEM further education learning aims being studied"
    ),

    make_equivalence_pair(
        "Rise in enrolments across all subjects",
        "Rise in full-time degree enrolments in all subjects"
    ),

    make_equivalence_pair(
        "Overall fall in part-time degree enrolments",
        "Fall in part-time degree enrolments overall"
    ),

    make_equivalence_pair(
        "People who graduated with a STEM degree",
        "People graduating with a STEM degree"
    ),

    make_equivalence_pair(
        "STEM graduates known to be working in a STEM occupation within six months",
        "STEM graduates known to be working in a STEM occupation"
    ),

    make_equivalence_pair(
        "STEM graduates whose destinations were unknown",
        "STEM graduates whose destinations are unknown"
    ),

    make_equivalence_pair(
        "T levels are designed to improve vocational education by standardising qualifications, aligning syllabuses with employer demand and establishing clear routes into careers",
        "Clear routes into careers established by T levels"
    ),

    make_equivalence_pair(
        "Institutes of technology will target skills gaps at levels 4 and upwards, particularly in STEM areas",
        "Qualification level targeted by institutes of technology"
    ),

    make_equivalence_pair(
        "Target for recruiting additional maths and physics teachers",
        "Target for recruiting additional teachers"
    ),

    make_equivalence_pair(
        "Returning teachers recruited by the return to teaching pilot",
        "Returning teachers recruited"
    ),

    make_equivalence_pair(
        "Target for returning teachers recruited by the return to teaching pilot",
        "Recruitment target"
    ),

    make_equivalence_pair(
        "Spent on, or committed to, key STEM-specific interventions",
        "Spending on or committed to key STEM-specific interventions"
    ),

    make_equivalence_pair(
        "Graduates in STEM subjects known to be working in a STEM occupation six months later",
        "STEM graduates known to be working in a STEM occupation"
    ),

    make_equivalence_pair(
        "Additional STEM technicians estimated as needed to meet employer demand",
        "Additional STEM technicians estimated to be needed to meet employer demand"
    ),

    make_equivalence_pair(
        "Rise in STEM A level examination entries compared with the previous year",
        "Rise in STEM A level examination entries"
    ),

    make_equivalence_pair(
        "There is no universally accepted definition of STEM in either education or employment",
        "Definition of STEM"
    ),

    make_equivalence_pair(
        "DfE is responsible for the majority of STEM skills interventions and BEIS has a cross-cutting role",
        "Departmental responsibilities for STEM skills"
    ),

    make_equivalence_pair(
        "The absence of a precise understanding of the STEM skills problem means the efforts of DfE and BEIS are not well prioritised and a better targeted approach is needed to demonstrate value for money",
        "DfE and BEIS need a shared vision, coordinated plans and a better targeted approach to demonstrate value for money"
    ),

    make_equivalence_pair(
        "The impact of exit from the EU is difficult to predict",
        "Impact of exit from the EU is difficult to predict"
    ),

    make_equivalence_pair(
        "Government does not have a stable and consistent set of definitions for STEM in either an educational or a work context",
        "Government does not have a stable and consistent set of definitions for STEM"
    ),

    make_equivalence_pair(
        "Configure the labour market intelligence generated by Skills Advisory Panels and other mechanisms so that it enables effective decision-making",
        "Configure labour market intelligence so that it enables effective decision-making"
    ),

    make_equivalence_pair(
        "Provide departments with clarity on the different STEM definitions used in different contexts, and reasons for these different definitions",
        "Provide departments with clarity on the different STEM definitions used in different contexts and reasons for these different definitions"
    ),

    make_equivalence_pair(
        "Strengthen its work to evaluate and identify what is effective in its activities to promote participation in STEM education and skills development, and ensure this is shared with its delivery partners",
        "Strengthen work to evaluate and identify what is effective in activities to promote participation in STEM education and skills development and share this with delivery partners"
    ),

    make_equivalence_pair(
        "Working with other departments, use data on skills mismatches resulting from EU exit to establish the position across relevant sectors and determine whether key capabilities are at risk",
        "Use data on skills mismatches resulting from EU exit to establish the position across relevant sectors and determine whether key capabilities are at risk"
    ),

    # Previously missing Policy context record:
    make_equivalence_pair(
        "The key routes for developing STEM knowledge and skills are schools and sixth-form colleges, further education colleges, apprenticeships and higher education institutions",
        "Main STEM skills-development routes"
    )
}


def canonical_metric(value):
    return normalise_text(value)


def is_metric_equivalent(
    reference_value,
    extracted_value
):
    if (
        canonical_metric(reference_value)
        == canonical_metric(extracted_value)
    ):
        return True

    return pair_is_controlled_equivalent(
        reference_value,
        extracted_value,
        CONTROLLED_METRIC_EQUIVALENCE_PAIRS
    )


# ------------------------------------------------------------
# 12.7 Topic equivalence
# ------------------------------------------------------------

CONTROLLED_TOPIC_EQUIVALENCE_PAIRS = {
    make_equivalence_pair(
        "STEM A level initiatives",
        "A levels"
    ),

    make_equivalence_pair(
        "Female participation in STEM A levels",
        "Female students — STEM A level exam entries"
    ),

    make_equivalence_pair(
        "Female participation in computing",
        "Female students — computing"
    ),

    make_equivalence_pair(
        "Female participation in physics",
        "Female students — physics"
    ),

    make_equivalence_pair(
        "Female participation in mathematics",
        "Female students — mathematics"
    ),

    make_equivalence_pair(
        "Female participation in STEM apprenticeships",
        "Female apprentices — STEM apprenticeships"
    ),

    make_equivalence_pair(
        "Female participation in all apprenticeships",
        "Female apprentices — all apprenticeship starts"
    ),

    make_equivalence_pair(
        "Female participation in undergraduate STEM",
        "Female students — undergraduate STEM courses"
    ),

    make_equivalence_pair(
        "Female participation in all undergraduate courses",
        "Female students — all enrolments"
    ),

    make_equivalence_pair(
        "Further education STEM participation",
        "STEM further education learning aims"
    ),

    make_equivalence_pair(
        "Full-time undergraduate STEM participation",
        "Full-time STEM degrees"
    ),

    make_equivalence_pair(
        "All-subject undergraduate participation",
        "Full-time degrees — all subjects"
    ),

    make_equivalence_pair(
        "Part-time undergraduate STEM participation",
        "Part-time undergraduate STEM courses"
    ),

    make_equivalence_pair(
        "Part-time undergraduate participation",
        "Part-time degree enrolments"
    ),

    make_equivalence_pair(
        "STEM graduate destinations",
        "STEM graduates"
    ),

    make_equivalence_pair(
        "T levels",
        "Technical levels (T levels)"
    ),

    make_equivalence_pair(
        "National colleges",
        "National colleges programme"
    ),

    make_equivalence_pair(
        "Teacher supply",
        "Maths and physics teacher supply"
    ),

    make_equivalence_pair(
        "Teacher recruitment",
        "Maths and physics teachers"
    ),

    make_equivalence_pair(
        "Teacher development",
        "Non-specialist maths and physics teachers"
    ),

    make_equivalence_pair(
        "STEM interventions",
        "Key STEM-specific interventions"
    ),

    make_equivalence_pair(
        "Undergraduate STEM participation",
        "Undergraduate STEM subjects"
    ),

    make_equivalence_pair(
        "STEM graduate destinations",
        "Graduates in STEM subjects"
    ),

    make_equivalence_pair(
        "STEM technician demand",
        "STEM technicians"
    ),

    make_equivalence_pair(
        "Female participation in STEM apprenticeships",
        "Women starting STEM apprenticeships"
    ),

    make_equivalence_pair(
        "STEM A level entries",
        "STEM A level examination entries"
    ),

    make_equivalence_pair(
        "Part-time undergraduate STEM participation",
        "Part-time undergraduate STEM degrees"
    ),

    make_equivalence_pair(
        "Definition of STEM",
        "STEM in education and employment"
    ),

    make_equivalence_pair(
        "Departmental responsibility",
        "DfE, BEIS and other government departments"
    ),

    make_equivalence_pair(
        "Value for money",
        "Government approach to improving STEM skills"
    ),

    make_equivalence_pair(
        "Labour market intelligence",
        "STEM skills intelligence"
    ),

    make_equivalence_pair(
        "Cross-government coordination",
        "Government coordination on STEM"
    ),

    make_equivalence_pair(
        "EU exit and STEM skills",
        "EU exit and availability of STEM skills"
    ),

    make_equivalence_pair(
        "Estimates of STEM skills needs",
        "STEM skills estimates"
    ),

    make_equivalence_pair(
        "STEM definitions",
        "STEM definitions in educational and work contexts"
    ),

    make_equivalence_pair(
        "Labour market intelligence",
        "Skills Advisory Panels and other labour market intelligence mechanisms"
    ),

    make_equivalence_pair(
        "Evaluation and knowledge sharing",
        "STEM education and skills development activities"
    ),

    make_equivalence_pair(
        "EU exit and STEM skills",
        "EU exit and skills mismatches"
    ),

    make_equivalence_pair(
        "Skills marketplace",
        "STEM skills marketplace"
    ),

    make_equivalence_pair(
        "Cross-government coordination",
        "Cross-government approach to STEM"
    ),

    # Previously missing Policy context record:
    make_equivalence_pair(
        "STEM skills pipeline",
        "Schools and sixth-form colleges; further education colleges; apprenticeships; higher education institutions"
    )
}


def canonical_topic(value):
    return normalise_text(value)


def is_topic_equivalent(
    reference_value,
    extracted_value
):
    if (
        canonical_topic(reference_value)
        == canonical_topic(extracted_value)
    ):
        return True

    return pair_is_controlled_equivalent(
        reference_value,
        extracted_value,
        CONTROLLED_TOPIC_EQUIVALENCE_PAIRS
    )


print(
    "Frozen D7 source-grounded equivalence rules loaded."
)

print(
    "Metric equivalence pairs:",
    len(CONTROLLED_METRIC_EQUIVALENCE_PAIRS)
)

print(
    "Topic equivalence pairs:",
    len(CONTROLLED_TOPIC_EQUIVALENCE_PAIRS)
)

Frozen D7 source-grounded equivalence rules loaded.
Metric equivalence pairs: 33
Topic equivalence pairs: 41


In [41]:
# ============================================================
# 12. Null-safe numeric comparison
# ============================================================

def numeric_value(value):
    if is_null(value) or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    if isinstance(value, str):
        text = (
            value.strip()
            .replace(",", "")
            .replace("−", "-")
            .replace("–", "-")
        )

        if re.fullmatch(
            r"-?\d+(?:\.\d+)?",
            text
        ):
            return float(text)

    return None


def numeric_exact_match(
    reference_value,
    extracted_value,
    tolerance=1e-9
):
    reference_number = numeric_value(reference_value)
    extracted_number = numeric_value(extracted_value)

    if (
        reference_number is None
        and extracted_number is None
    ):
        return True

    if (
        reference_number is None
        or extracted_number is None
    ):
        return False

    return math.isclose(
        reference_number,
        extracted_number,
        rel_tol=tolerance,
        abs_tol=tolerance
    )


def numeric_absolute_match(
    reference_value,
    extracted_value,
    tolerance=1e-9
):
    reference_number = numeric_value(reference_value)
    extracted_number = numeric_value(extracted_value)

    if (
        reference_number is None
        and extracted_number is None
    ):
        return True

    if (
        reference_number is None
        or extracted_number is None
    ):
        return False

    return math.isclose(
        abs(reference_number),
        abs(extracted_number),
        rel_tol=tolerance,
        abs_tol=tolerance
    )

In [42]:
# ============================================================
# 13. Prepare comparison copies and matching blocks
# ============================================================
# FROZEN FROM FINAL D7 BRANCH A VALIDATION — REUSED UNCHANGED.

reference_comparison_df = reference_df.copy(deep=True)
extracted_comparison_df = extracted_df.copy(deep=True)

reference_comparison_df["_reference_index"] = range(
    len(reference_comparison_df)
)

for dataframe in [
    reference_comparison_df,
    extracted_comparison_df
]:
    dataframe["_block_category"] = (
        dataframe["Category"]
        .map(normalise_text)
    )

    dataframe["_block_source"] = (
        dataframe["Source Location"]
        .map(normalise_text)
    )

    dataframe["_matching_block"] = list(
        zip(
            dataframe["_block_category"],
            dataframe["_block_source"]
        )
    )

print("Comparison copies prepared.")

Comparison copies prepared.


In [43]:
# ============================================================
# 14. Identity-only matching score
# ============================================================
# FROZEN FROM FINAL D7 BRANCH A VALIDATION — REUSED UNCHANGED.

def matching_score(
    reference_row,
    extracted_row
):
    # --------------------------------------------------------
    # Metric
    # --------------------------------------------------------
    # A predefined D7 source-grounded equivalence receives
    # full identity credit. Otherwise lexical similarity is
    # used only to support alignment.
    if is_metric_equivalent(
        reference_row["Metric"],
        extracted_row["Metric"]
    ):
        metric_score = 1.0

    else:
        metric_score = text_similarity(
            reference_row["Metric"],
            extracted_row["Metric"]
        )

    # --------------------------------------------------------
    # Topic
    # --------------------------------------------------------
    if is_topic_equivalent(
        reference_row["Topic"],
        extracted_row["Topic"]
    ):
        topic_score = 1.0

    else:
        topic_score = text_similarity(
            reference_row["Topic"],
            extracted_row["Topic"]
        )

    # --------------------------------------------------------
    # Statement / Section
    # --------------------------------------------------------
    if is_statement_equivalent(
        reference_row["Statement or Section"],
        extracted_row["Statement or Section"],
        reference_row["Source Location"]
    ):
        statement_score = 1.0

    else:
        statement_score = text_similarity(
            reference_row["Statement or Section"],
            extracted_row["Statement or Section"]
        )

    # --------------------------------------------------------
    # Reporting Period
    # --------------------------------------------------------
    if is_period_equivalent(
        reference_row["Reporting Period"],
        extracted_row["Reporting Period"]
    ):
        period_score = 1.0

    else:
        period_score = text_similarity(
            reference_row["Reporting Period"],
            extracted_row["Reporting Period"]
        )

    # --------------------------------------------------------
    # Weighted identity score
    # --------------------------------------------------------
    total_score = (
        MATCHING_WEIGHTS["metric"]
        * metric_score

        + MATCHING_WEIGHTS["topic"]
        * topic_score

        + MATCHING_WEIGHTS[
            "statement_or_section"
        ]
        * statement_score

        + MATCHING_WEIGHTS[
            "reporting_period"
        ]
        * period_score
    )

    return {
        "total": total_score,
        "metric": metric_score,
        "topic": topic_score,
        "statement": statement_score,
        "period": period_score
    }


print(
    "Matching uses Category + Source Location blocking "
    "and identity/context fields only."
)

print(
    "Value, Unit and Qualifier are excluded from alignment."
)

print(
    "Controlled D7 Metric, Topic and Statement equivalences "
    "may support identity matching but do not use outcome values."
)

Matching uses Category + Source Location blocking and identity/context fields only.
Value, Unit and Qualifier are excluded from alignment.
Controlled D7 Metric, Topic and Statement equivalences may support identity matching but do not use outcome values.


In [44]:
# ============================================================
# 15. One-to-one Hungarian record alignment
# ============================================================
# FROZEN FROM FINAL D7 BRANCH A VALIDATION — REUSED UNCHANGED.

all_blocks = sorted(
    set(
        reference_comparison_df[
            "_matching_block"
        ]
    )
    | set(
        extracted_comparison_df[
            "_matching_block"
        ]
    ),
    key=str
)

matched_pairs = []

unmatched_reference_indices = set(
    reference_comparison_df[
        "_reference_index"
    ].tolist()
)

unmatched_extraction_indices = set(
    extracted_comparison_df[
        "_extraction_index"
    ].tolist()
)

for block in all_blocks:

    reference_block = (
        reference_comparison_df.loc[
            reference_comparison_df[
                "_matching_block"
            ] == block
        ]
    )

    extraction_block = (
        extracted_comparison_df.loc[
            extracted_comparison_df[
                "_matching_block"
            ] == block
        ]
    )

    if reference_block.empty or extraction_block.empty:
        continue

    reference_rows = [
        row
        for _, row in reference_block.iterrows()
    ]

    extraction_rows = [
        row
        for _, row in extraction_block.iterrows()
    ]

    score_matrix = []
    details_matrix = []

    for reference_row in reference_rows:
        score_row = []
        details_row = []

        for extracted_row in extraction_rows:
            details = matching_score(
                reference_row,
                extracted_row
            )

            score_row.append(details["total"])
            details_row.append(details)

        score_matrix.append(score_row)
        details_matrix.append(details_row)

    cost_matrix = [
        [
            1.0 - score
            for score in row
        ]
        for row in score_matrix
    ]

    row_positions, column_positions = (
        linear_sum_assignment(cost_matrix)
    )

    for row_position, column_position in zip(
        row_positions,
        column_positions
    ):
        details = details_matrix[
            row_position
        ][
            column_position
        ]

        if details["total"] < MATCH_SCORE_THRESHOLD:
            continue

        reference_index = int(
            reference_rows[
                row_position
            ][
                "_reference_index"
            ]
        )

        extraction_index = int(
            extraction_rows[
                column_position
            ][
                "_extraction_index"
            ]
        )

        matched_pairs.append(
            {
                "reference_index": reference_index,
                "extraction_index": extraction_index,
                "matching_score": details["total"],
                "metric_matching_score": details["metric"],
                "topic_matching_score": details["topic"],
                "statement_matching_score": details["statement"],
                "period_matching_score": details["period"]
            }
        )

        unmatched_reference_indices.discard(
            reference_index
        )

        unmatched_extraction_indices.discard(
            extraction_index
        )


aligned_record_count = len(matched_pairs)
missing_record_count = len(unmatched_reference_indices)
unsupported_record_count = len(unmatched_extraction_indices)

print("Aligned records:", aligned_record_count)
print("Missing records:", missing_record_count)
print("Unsupported/unmatched extracted records:", unsupported_record_count)

Aligned records: 59
Missing records: 0
Unsupported/unmatched extracted records: 1


In [45]:
# ============================================================
# 16. Missing and unsupported/unmatched record tables
# ============================================================

missing_records_df = (
    reference_comparison_df.loc[
        reference_comparison_df[
            "_reference_index"
        ].isin(
            unmatched_reference_indices
        ),
        FIELDS
    ].copy()
)

unsupported_records_df = (
    extracted_comparison_df.loc[
        extracted_comparison_df[
            "_extraction_index"
        ].isin(
            unmatched_extraction_indices
        ),
        FIELDS + ["_extraction_index"]
    ].copy()
)

print("Missing:", len(missing_records_df))
print("Unsupported/unmatched:", len(unsupported_records_df))

if not missing_records_df.empty:
    display(missing_records_df)

if not unsupported_records_df.empty:
    display(unsupported_records_df)

Missing: 0
Unsupported/unmatched: 1


,Category,Statement or Section,Metric,Topic,Value,Unit,Qualifier,Reporting Period,Source Location,_extraction_index
43,Education pipeline statistic,The performance of the education pipeline in d...,Graduate outcomes data collection point,Graduate outcomes data,NaN,None,None,between 12 and 18 months after graduation,PDF page 11 — Summary paragraph 17,43


In [46]:
# ============================================================
# 17. Field-level comparison of aligned records
# ============================================================
# FROZEN FROM FINAL D7 BRANCH A VALIDATION — REUSED UNCHANGED.

def exact_text_match(
    first,
    second
):
    return (
        normalise_text(first)
        == normalise_text(second)
    )


comparison_rows = []

for pair in matched_pairs:

    reference_row = (
        reference_comparison_df.loc[
            reference_comparison_df[
                "_reference_index"
            ] == pair["reference_index"]
        ].iloc[0]
    )

    extracted_row = (
        extracted_comparison_df.loc[
            extracted_comparison_df[
                "_extraction_index"
            ] == pair["extraction_index"]
        ].iloc[0]
    )

    # --------------------------------------------------------
    # Diagnostic lexical similarities
    # --------------------------------------------------------

    metric_similarity = text_similarity(
        reference_row["Metric"],
        extracted_row["Metric"]
    )

    topic_similarity = text_similarity(
        reference_row["Topic"],
        extracted_row["Topic"]
    )

    # --------------------------------------------------------
    # Value comparison
    # --------------------------------------------------------

    value_match = numeric_exact_match(
        reference_row["Value"],
        extracted_row["Value"]
    )

    value_absolute_match = numeric_absolute_match(
        reference_row["Value"],
        extracted_row["Value"]
    )

    sign_only_difference = (
        not value_match
        and value_absolute_match
    )

    # --------------------------------------------------------
    # Controlled D7 field comparison
    # --------------------------------------------------------

    field_matches = {
        "Category": exact_text_match(
            reference_row["Category"],
            extracted_row["Category"]
        ),

        "Statement or Section":
            is_statement_equivalent(
                reference_row[
                    "Statement or Section"
                ],
                extracted_row[
                    "Statement or Section"
                ],
                reference_row[
                    "Source Location"
                ]
            ),

        "Metric": is_metric_equivalent(
            reference_row["Metric"],
            extracted_row["Metric"]
        ),

        "Topic": is_topic_equivalent(
            reference_row["Topic"],
            extracted_row["Topic"]
        ),

        "Value": value_match,

        "Unit": is_unit_equivalent(
            reference_row["Unit"],
            extracted_row["Unit"]
        ),

        "Qualifier": is_qualifier_equivalent(
            reference_row["Qualifier"],
            extracted_row["Qualifier"]
        ),

        "Reporting Period":
            is_period_equivalent(
                reference_row[
                    "Reporting Period"
                ],
                extracted_row[
                    "Reporting Period"
                ]
            ),

        "Source Location": exact_text_match(
            reference_row[
                "Source Location"
            ],
            extracted_row[
                "Source Location"
            ]
        )
    }

    # --------------------------------------------------------
    # Discrepancy classification
    # --------------------------------------------------------

    all_mismatched_fields = [
        field
        for field in FIELDS
        if not field_matches[field]
    ]

    primary_mismatched_fields = [
        field
        for field in PRIMARY_CORRECTNESS_FIELDS
        if not field_matches[field]
    ]

    fully_correct = (
        len(primary_mismatched_fields) == 0
    )

    # --------------------------------------------------------
    # Identify where controlled equivalence was required
    # --------------------------------------------------------

    metric_rule_applied = (
        not exact_text_match(
            reference_row["Metric"],
            extracted_row["Metric"]
        )
        and is_metric_equivalent(
            reference_row["Metric"],
            extracted_row["Metric"]
        )
    )

    topic_rule_applied = (
        not exact_text_match(
            reference_row["Topic"],
            extracted_row["Topic"]
        )
        and is_topic_equivalent(
            reference_row["Topic"],
            extracted_row["Topic"]
        )
    )

    statement_rule_applied = (
        not (
            canonical_statement(
                reference_row[
                    "Statement or Section"
                ]
            )
            ==
            canonical_statement(
                extracted_row[
                    "Statement or Section"
                ]
            )
        )
        and is_statement_equivalent(
            reference_row[
                "Statement or Section"
            ],
            extracted_row[
                "Statement or Section"
            ],
            reference_row[
                "Source Location"
            ]
        )
    )

    unit_rule_applied = (
        not (
            canonical_unit(
                reference_row["Unit"]
            )
            ==
            canonical_unit(
                extracted_row["Unit"]
            )
        )
        and is_unit_equivalent(
            reference_row["Unit"],
            extracted_row["Unit"]
        )
    )

    qualifier_rule_applied = (
        not (
            canonical_qualifier(
                reference_row["Qualifier"]
            )
            ==
            canonical_qualifier(
                extracted_row["Qualifier"]
            )
        )
        and is_qualifier_equivalent(
            reference_row["Qualifier"],
            extracted_row["Qualifier"]
        )
    )

    period_rule_applied = (
        normalise_text(
            reference_row[
                "Reporting Period"
            ]
        )
        !=
        normalise_text(
            extracted_row[
                "Reporting Period"
            ]
        )
        and is_period_equivalent(
            reference_row[
                "Reporting Period"
            ],
            extracted_row[
                "Reporting Period"
            ]
        )
    )

    # --------------------------------------------------------
    # Detailed output row
    # --------------------------------------------------------

    output_row = {
        "Reference Index":
            pair["reference_index"],

        "Extraction Index":
            pair["extraction_index"],

        # Required by downstream category metrics.
        "Category":
            reference_row["Category"],

        "Matching Score":
            pair["matching_score"],

        "Metric Matching Score":
            pair["metric_matching_score"],

        "Topic Matching Score":
            pair["topic_matching_score"],

        "Statement Matching Score":
            pair["statement_matching_score"],

        "Reporting Period Matching Score":
            pair["period_matching_score"],

        "Metric Lexical Similarity":
            metric_similarity,

        "Topic Lexical Similarity":
            topic_similarity,

        "Metric Equivalence Rule Applied":
            bool(metric_rule_applied),

        "Topic Equivalence Rule Applied":
            bool(topic_rule_applied),

        "Statement Equivalence Rule Applied":
            bool(statement_rule_applied),

        "Unit Equivalence Rule Applied":
            bool(unit_rule_applied),

        "Qualifier Equivalence Rule Applied":
            bool(qualifier_rule_applied),

        "Reporting Period Equivalence Rule Applied":
            bool(period_rule_applied),

        "Value Sign-Only Difference":
            bool(sign_only_difference),

        "Fully Correct":
            bool(fully_correct),

        "all_mismatched_fields":
            ", ".join(
                all_mismatched_fields
            ),

        "primary_mismatched_fields":
            ", ".join(
                primary_mismatched_fields
            )
    }

    # --------------------------------------------------------
    # Preserve reference/extracted values + match flags
    # --------------------------------------------------------

    for field in FIELDS:

        output_row[
            f"Reference {field}"
        ] = reference_row[field]

        output_row[
            f"Extracted {field}"
        ] = extracted_row[field]

        output_row[
            f"{field} Match"
        ] = bool(
            field_matches[field]
        )

    comparison_rows.append(
        output_row
    )


comparison_df = pd.DataFrame(
    comparison_rows
)

print(
    "Compared aligned records:",
    len(comparison_df)
)

print(
    "Fully correct:",
    int(
        comparison_df[
            "Fully Correct"
        ].sum()
    )
    if not comparison_df.empty
    else 0
)

print(
    "Sign-only value differences:",
    int(
        comparison_df[
            "Value Sign-Only Difference"
        ].sum()
    )
    if not comparison_df.empty
    else 0
)

if not comparison_df.empty:

    print(
        "Metric equivalence rules applied:",
        int(
            comparison_df[
                "Metric Equivalence Rule Applied"
            ].sum()
        )
    )

    print(
        "Topic equivalence rules applied:",
        int(
            comparison_df[
                "Topic Equivalence Rule Applied"
            ].sum()
        )
    )

    print(
        "Statement equivalence rules applied:",
        int(
            comparison_df[
                "Statement Equivalence Rule Applied"
            ].sum()
        )
    )

    print(
        "Unit equivalence rules applied:",
        int(
            comparison_df[
                "Unit Equivalence Rule Applied"
            ].sum()
        )
    )

    print(
        "Qualifier equivalence rules applied:",
        int(
            comparison_df[
                "Qualifier Equivalence Rule Applied"
            ].sum()
        )
    )

    print(
        "Reporting-period equivalence rules applied:",
        int(
            comparison_df[
                "Reporting Period Equivalence Rule Applied"
            ].sum()
        )
    )

display(
    comparison_df.head(10)
)

Compared aligned records: 59
Fully correct: 11
Sign-only value differences: 0
Metric equivalence rules applied: 19
Topic equivalence rules applied: 25
Statement equivalence rules applied: 20
Unit equivalence rules applied: 4
Qualifier equivalence rules applied: 1
Reporting-period equivalence rules applied: 11


,Reference Index,Extraction Index,Category,Matching Score,Metric Matching Score,Topic Matching Score,Statement Matching Score,Reporting Period Matching Score,Metric Lexical Similarity,Topic Lexical Similarity,...,Unit Match,Reference Qualifier,Extracted Qualifier,Qualifier Match,Reference Reporting Period,Extracted Reporting Period,Reporting Period Match,Reference Source Location,Extracted Source Location,Source Location Match
0,20,19,Education pipeline statistic,1.000000,1.0,1.000000,1.000000,1.0,1.000000,0.500000,...,True,None,None,True,2011/12 to 2016/17,between 2011/12 and 2016/17,True,PDF page 10 — Summary paragraph 13,PDF page 10 — Summary paragraph 13,True
1,21,20,Education pipeline statistic,0.839976,1.0,0.753623,0.425926,1.0,0.810811,0.753623,...,True,None,only,False,2016/17,2016/17,True,PDF page 10 — Summary paragraph 14,PDF page 10 — Summary paragraph 14,True
2,22,21,Education pipeline statistic,0.830282,1.0,0.721311,0.425926,1.0,0.865385,0.721311,...,True,None,just,False,2016/17,2016/17,True,PDF page 10 — Summary paragraph 14,PDF page 10 — Summary paragraph 14,True
3,23,22,Education pipeline statistic,0.824415,1.0,0.701754,0.425926,1.0,0.860000,0.701754,...,True,None,None,True,2016/17,2016/17,True,PDF page 10 — Summary paragraph 14,PDF page 10 — Summary paragraph 14,True
4,24,23,Education pipeline statistic,0.835427,1.0,0.738462,0.425926,1.0,0.870370,0.738462,...,True,None,None,True,2016/17,2016/17,True,PDF page 10 — Summary paragraph 14,PDF page 10 — Summary paragraph 14,True
5,25,24,Education pipeline statistic,0.853889,1.0,0.800000,0.425926,1.0,0.857143,0.800000,...,True,around,around,True,2016/17,2016/17,True,PDF page 10 — Summary paragraph 14,PDF page 10 — Summary paragraph 14,True
6,26,25,Education pipeline statistic,0.852245,1.0,0.794521,0.425926,1.0,0.860465,0.794521,...,True,more than,more than,True,2016/17,2016/17,True,PDF page 10 — Summary paragraph 14,PDF page 10 — Summary paragraph 14,True
7,27,26,Education pipeline statistic,0.826547,1.0,0.708861,0.425926,1.0,0.865385,0.708861,...,True,around,around,True,None,None,True,PDF page 10 — Summary paragraph 14,PDF page 10 — Summary paragraph 14,True
8,28,27,Education pipeline statistic,0.818434,1.0,0.681818,0.425926,1.0,0.869565,0.681818,...,True,more than,more than,True,None,None,True,PDF page 10 — Summary paragraph 14,PDF page 10 — Summary paragraph 14,True
9,29,28,Education pipeline statistic,0.913889,1.0,1.000000,0.425926,1.0,1.000000,1.000000,...,True,None,None,True,2012/13,2012/13,True,PDF page 10 — Summary paragraph 15,PDF page 10 — Summary paragraph 15,True


In [47]:
# ============================================================
# 18. Split fully correct and discrepant aligned records
# ============================================================

if comparison_df.empty:
    fully_correct_records_df = comparison_df.copy()
    discrepant_records_df = comparison_df.copy()

else:
    fully_correct_records_df = (
        comparison_df.loc[
            comparison_df["Fully Correct"]
        ].copy()
    )

    discrepant_records_df = (
        comparison_df.loc[
            ~comparison_df["Fully Correct"]
        ].copy()
    )

fully_correct_record_count = len(
    fully_correct_records_df
)

discrepant_record_count = len(
    discrepant_records_df
)

print("Fully correct records:", fully_correct_record_count)
print("Discrepant aligned records:", discrepant_record_count)

if not discrepant_records_df.empty:
    display(
        discrepant_records_df[
            [
                "Reference Index",
                "Extraction Index",
                "Reference Metric",
                "Extracted Metric",
                "Metric Lexical Similarity",
                "Reference Topic",
                "Extracted Topic",
                "Topic Lexical Similarity",
                "Value Sign-Only Difference",
                "primary_mismatched_fields"
            ]
        ]
    )

Fully correct records: 11
Discrepant aligned records: 48


,Reference Index,Extraction Index,Reference Metric,Extracted Metric,Metric Lexical Similarity,Reference Topic,Extracted Topic,Topic Lexical Similarity,Value Sign-Only Difference,primary_mismatched_fields
1,21,20,Female students as a share of all STEM A level...,Female share of all STEM A level exam entries,0.810811,Female participation in STEM A levels,Female students in STEM A levels,0.753623,False,"Statement or Section, Topic, Qualifier"
2,22,21,Female students as a share of computing examin...,Female share of computing examination entries,0.865385,Female participation in computing,Female students in computing,0.721311,False,"Statement or Section, Topic, Qualifier"
3,23,22,Female students as a share of physics examinat...,Female share of physics examination entries,0.860000,Female participation in physics,Female students in physics,0.701754,False,"Statement or Section, Topic"
4,24,23,Female students as a share of mathematics exam...,Female share of mathematics examination entries,0.870370,Female participation in mathematics,Female students in mathematics,0.738462,False,"Statement or Section, Topic"
5,25,24,Female students as a share of STEM apprentices...,Female share of STEM apprenticeship starts,0.857143,Female participation in STEM apprenticeships,Females in STEM apprenticeships,0.800000,False,"Statement or Section, Topic"
6,26,25,Women as a share of all apprenticeship starts,Female share of all apprenticeship starts,0.860465,Female participation in all apprenticeships,Females in all apprenticeships,0.794521,False,"Statement or Section, Topic"
7,27,26,Female students as a share of undergraduate ST...,Female share of undergraduate STEM enrolments,0.865385,Female participation in undergraduate STEM,Females in undergraduate STEM courses,0.708861,False,"Statement or Section, Topic"
8,28,27,Women as a share of all undergraduate enrolments,Female share of all undergraduate enrolments,0.869565,Female participation in all undergraduate courses,Females in all undergraduate enrolments,0.681818,False,"Statement or Section, Topic"
9,29,28,STEM apprenticeship starts,STEM apprenticeship starts,1.000000,STEM apprenticeships,STEM apprenticeships,1.000000,False,Statement or Section
10,30,29,STEM apprenticeship starts,STEM apprenticeship starts,1.000000,STEM apprenticeships,STEM apprenticeships,1.000000,False,Statement or Section


In [48]:
# ============================================================
# 19. Field-level validation and error summary
# ============================================================

field_validation_rows = []

for field in FIELDS:

    match_column = f"{field} Match"

    correct_count = (
        int(
            comparison_df[
                match_column
            ].sum()
        )
        if not comparison_df.empty
        else 0
    )

    aligned_count = len(comparison_df)

    accuracy = (
        correct_count / aligned_count
        if aligned_count
        else None
    )

    field_validation_rows.append(
        {
            "Field": field,
            "Aligned Records": aligned_count,
            "Correct": correct_count,
            "Incorrect": aligned_count - correct_count,
            "Accuracy": accuracy
        }
    )


field_validation_df = pd.DataFrame(
    field_validation_rows
)

field_error_summary_df = (
    field_validation_df[
        [
            "Field",
            "Incorrect",
            "Accuracy"
        ]
    ].copy()
)

display(field_validation_df)

,Field,Aligned Records,Correct,Incorrect,Accuracy
0,Category,59,59,0,1.000000
1,Statement or Section,59,30,29,0.508475
2,Metric,59,37,22,0.627119
3,Topic,59,40,19,0.677966
4,Value,59,59,0,1.000000
5,Unit,59,59,0,1.000000
6,Qualifier,59,54,5,0.915254
7,Reporting Period,59,57,2,0.966102
8,Source Location,59,59,0,1.000000


In [49]:
# ============================================================
# 20. Common record-level validation metrics
# ============================================================

reference_record_count = len(reference_df)
extracted_record_count = len(extracted_df)

completeness = (
    aligned_record_count
    / reference_record_count
    if reference_record_count
    else 0.0
)

missing_rate = (
    missing_record_count
    / reference_record_count
    if reference_record_count
    else 0.0
)

record_precision_exact = (
    fully_correct_record_count
    / extracted_record_count
    if extracted_record_count
    else 0.0
)

record_recall_exact = (
    fully_correct_record_count
    / reference_record_count
    if reference_record_count
    else 0.0
)

record_f1_exact = (
    2
    * record_precision_exact
    * record_recall_exact
    / (
        record_precision_exact
        + record_recall_exact
    )
    if (
        record_precision_exact
        + record_recall_exact
    )
    else 0.0
)

unsupported_rate = (
    unsupported_record_count
    / extracted_record_count
    if extracted_record_count
    else 0.0
)

discrepancy_rate_among_aligned = (
    discrepant_record_count
    / aligned_record_count
    if aligned_record_count
    else 0.0
)

primary_field_match_count = 0
primary_field_comparison_count = (
    aligned_record_count
    * len(PRIMARY_CORRECTNESS_FIELDS)
)

if not comparison_df.empty:
    for field in PRIMARY_CORRECTNESS_FIELDS:
        primary_field_match_count += int(
            comparison_df[
                f"{field} Match"
            ].sum()
        )

overall_primary_field_accuracy = (
    primary_field_match_count
    / primary_field_comparison_count
    if primary_field_comparison_count
    else None
)

sign_only_value_difference_count = (
    int(
        comparison_df[
            "Value Sign-Only Difference"
        ].sum()
    )
    if not comparison_df.empty
    else 0
)

print("Completeness:", completeness)
print("Exact precision:", record_precision_exact)
print("Exact recall:", record_recall_exact)
print("Exact F1:", record_f1_exact)
print("Primary field accuracy:", overall_primary_field_accuracy)
print("Sign-only value differences:", sign_only_value_difference_count)

Completeness: 1.0
Exact precision: 0.18333333333333332
Exact recall: 0.1864406779661017
Exact F1: 0.18487394957983194
Primary field accuracy: 0.8549905838041432
Sign-only value differences: 0


In [50]:
# ============================================================
# 21. Category-level metrics
# ============================================================

category_metric_rows = []

all_categories = sorted(
    set(EXPECTED_CATEGORY_COUNTS)
    | set(
        extracted_df[
            "Category"
        ].dropna().tolist()
    )
)

for category in all_categories:

    expected_records = int(
        (
            reference_df["Category"]
            == category
        ).sum()
    )

    extracted_records_in_category = int(
        (
            extracted_df["Category"]
            == category
        ).sum()
    )

    category_comparison = (
        comparison_df.loc[
            comparison_df["Reference Category"]
            == category
        ]
        if not comparison_df.empty
        else comparison_df
    )

    aligned_records_in_category = len(
        category_comparison
    )

    fully_correct_in_category = (
        int(
            category_comparison[
                "Fully Correct"
            ].sum()
        )
        if not category_comparison.empty
        else 0
    )

    discrepant_in_category = (
        aligned_records_in_category
        - fully_correct_in_category
    )

    category_completeness = (
        aligned_records_in_category
        / expected_records
        if expected_records
        else None
    )

    category_precision = (
        fully_correct_in_category
        / extracted_records_in_category
        if extracted_records_in_category
        else 0.0
    )

    category_recall = (
        fully_correct_in_category
        / expected_records
        if expected_records
        else 0.0
    )

    category_f1 = (
        2
        * category_precision
        * category_recall
        / (
            category_precision
            + category_recall
        )
        if (
            category_precision
            + category_recall
        )
        else 0.0
    )

    category_metric_rows.append(
        {
            "Category": category,
            "Expected Records": expected_records,
            "Extracted Records": extracted_records_in_category,
            "Aligned Records": aligned_records_in_category,
            "Fully Correct Records": fully_correct_in_category,
            "Discrepant Records": discrepant_in_category,
            "Completeness": category_completeness,
            "Record Precision Exact": category_precision,
            "Record Recall Exact": category_recall,
            "Record F1 Exact": category_f1
        }
    )


category_metrics_df = pd.DataFrame(
    category_metric_rows
)

display(category_metrics_df)

,Category,Expected Records,Extracted Records,Aligned Records,Fully Correct Records,Discrepant Records,Completeness,Record Precision Exact,Record Recall Exact,Record F1 Exact
0,Education pipeline statistic,24,25,24,1,23,1.0,0.040000,0.041667,0.040816
1,Government initiative,9,9,9,2,7,1.0,0.222222,0.222222,0.222222
2,Key fact,10,10,10,7,3,1.0,0.700000,0.700000,0.700000
3,Policy context,3,3,3,1,2,1.0,0.333333,0.333333,0.333333
4,Policy finding,7,7,7,0,7,1.0,0.000000,0.000000,0.000000
5,Recommendation,6,6,6,0,6,1.0,0.000000,0.000000,0.000000


In [51]:
# ============================================================
# 22. Define schema validity independently from completeness
# ============================================================

schema_validity = all(
    [
        valid_json,
        top_level_object_valid,
        document_id_correct,
        branch_correct,
        records_is_list,
        record_schema_valid,
        field_types_valid,
        branch_c_structure_valid
    ]
)

schema_diagnostics = {
    "valid_json": bool(valid_json),
    "top_level_object_valid": bool(top_level_object_valid),
    "document_id_correct": bool(document_id_correct),
    "branch_correct": bool(branch_correct),
    "records_is_list": bool(records_is_list),
    "record_schema_valid": bool(record_schema_valid),
    "field_types_valid": bool(field_types_valid),
    "records_with_structure_issues": int(len(schema_issues_df)),
    "records_with_type_issues": int(len(type_issues_df)),
    "field_order_diagnostic_count": int(len(field_order_issues_df)),
    "branch_C_structure_valid": bool(branch_c_structure_valid),
    "branch_C_record_schema_valid":
        branch_c_structure_check.get("record_schema_valid"),
    "branch_C_field_types_valid":
        branch_c_structure_check.get("field_types_valid"),
    "schema_validity": bool(schema_validity)
}

content_diagnostics = {
    "reference_record_count_valid":
        bool(reference_record_count_valid),

    "reference_category_counts_valid":
        bool(reference_category_counts_valid),

    "extraction_record_count_valid":
        bool(extraction_record_count_valid),

    "extraction_category_counts_valid":
        bool(extraction_category_counts_valid),

    "mandatory_fields_complete":
        bool(mandatory_fields_complete),

    "branch_C_scope_complete":
        branch_c_structure_check.get("scope_complete"),

    "branch_C_content_diagnostics":
        branch_c_structure_check.get("content_diagnostics")
}

print("Schema validity:", schema_validity)
print(
    json.dumps(
        schema_diagnostics,
        ensure_ascii=False,
        indent=2
    )
)

print("\nContent diagnostics:")
print(
    json.dumps(
        content_diagnostics,
        ensure_ascii=False,
        indent=2
    )
)

Schema validity: True
{
  "valid_json": true,
  "top_level_object_valid": true,
  "document_id_correct": true,
  "branch_correct": true,
  "records_is_list": true,
  "record_schema_valid": true,
  "field_types_valid": true,
  "records_with_structure_issues": 0,
  "records_with_type_issues": 0,
  "field_order_diagnostic_count": 0,
  "branch_C_structure_valid": true,
  "branch_C_record_schema_valid": true,
  "branch_C_field_types_valid": true,
  "schema_validity": true
}

Content diagnostics:
{
  "reference_record_count_valid": true,
  "reference_category_counts_valid": true,
  "extraction_record_count_valid": false,
  "extraction_category_counts_valid": false,
  "mandatory_fields_complete": true,
  "branch_C_scope_complete": false,
  "branch_C_content_diagnostics": {
    "expected_record_count": 59,
    "observed_record_count": 60,
    "record_count_matches_reference": false,
    "expected_category_counts": {
      "Key fact": 10,
      "Policy context": 3,
      "Policy finding": 7,
  

In [52]:
# ============================================================
# 23. Validation summary table
# ============================================================

validation_summary_df = pd.DataFrame(
    [
        {
            "Metric": "Reference records",
            "Value": reference_record_count
        },
        {
            "Metric": "Extracted records",
            "Value": extracted_record_count
        },
        {
            "Metric": "Aligned records",
            "Value": aligned_record_count
        },
        {
            "Metric": "Fully correct records",
            "Value": fully_correct_record_count
        },
        {
            "Metric": "Discrepant records",
            "Value": discrepant_record_count
        },
        {
            "Metric": "Missing records",
            "Value": missing_record_count
        },
        {
            "Metric": "Unsupported extracted records",
            "Value": unsupported_record_count
        },
        {
            "Metric": "Completeness",
            "Value": completeness
        },
        {
            "Metric": "Record precision exact",
            "Value": record_precision_exact
        },
        {
            "Metric": "Record recall exact",
            "Value": record_recall_exact
        },
        {
            "Metric": "Record F1 exact",
            "Value": record_f1_exact
        },
        {
            "Metric": "Overall primary field accuracy",
            "Value": overall_primary_field_accuracy
        },
        {
            "Metric": "Sign-only value differences",
            "Value": sign_only_value_difference_count
        },
        {
            "Metric": "Schema valid",
            "Value": bool(schema_validity)
        }
    ]
)

display(validation_summary_df)

,Metric,Value
0,Reference records,59
1,Extracted records,60
2,Aligned records,59
3,Fully correct records,11
4,Discrepant records,48
5,Missing records,0
6,Unsupported extracted records,1
7,Completeness,1.0
8,Record precision exact,0.183333
9,Record recall exact,0.186441


In [53]:
# ============================================================
# 24. Preserve Branch C normalisation-integrity diagnostics
# ============================================================
# Stage 2 B→C representation integrity is reported separately from
# Stage 4 extraction correctness.

with NORMALISATION_INTEGRITY_PATH.open(
    "r",
    encoding="utf-8"
) as file:
    normalisation_integrity = json.load(file)

if normalisation_integrity.get("document_id") != DOCUMENT_ID:
    raise ValueError(
        "Normalisation-integrity document_id does not match D7."
    )

if normalisation_integrity.get("branch") != BRANCH:
    raise ValueError(
        "Normalisation-integrity branch does not match Branch C."
    )

if normalisation_integrity.get("parent_branch") != "B":
    raise ValueError(
        "Normalisation-integrity parent_branch does not match Branch B."
    )

representation_integrity = {
    "parent_branch":
        normalisation_integrity.get("parent_branch"),

    "parent_equivalence_passed":
        bool(
            normalisation_integrity.get(
                "parent_equivalence_passed",
                False
            )
        ),

    "normalisation_integrity_passed":
        bool(
            normalisation_integrity.get(
                "normalisation_integrity_passed",
                False
            )
        ),

    "page_sequence_preserved":
        normalisation_integrity.get(
            "page_sequence_preserved"
        ),

    "deterministic_representation_verified":
        normalisation_integrity.get(
            "deterministic_representation_verified"
        ),

    "all_expected_components_preserved":
        normalisation_integrity.get(
            "all_expected_components_preserved"
        ),

    "all_representative_content_preserved":
        normalisation_integrity.get(
            "all_representative_content_preserved"
        ),

    "numeric_values_preserved":
        normalisation_integrity.get(
            "numeric_values_preserved"
        ),

    "all_scope_pages_present":
        normalisation_integrity.get(
            "all_scope_pages_present"
        ),

    "complete_12_page_representation_retained":
        normalisation_integrity.get(
            "complete_12_page_representation_retained"
        ),

    "source_scope_filtering_applied":
        normalisation_integrity.get(
            "source_scope_filtering_applied"
        ),

    "page_removal_applied":
        normalisation_integrity.get(
            "page_removal_applied"
        ),

    "page_cropping_applied":
        normalisation_integrity.get(
            "page_cropping_applied"
        ),

    "ocr_applied":
        normalisation_integrity.get(
            "ocr_applied"
        ),

    "unicode_nfkc_normalisation_applied":
        normalisation_integrity.get(
            "unicode_nfkc_normalisation_applied"
        ),

    "unicode_space_standardisation_applied":
        normalisation_integrity.get(
            "unicode_space_standardisation_applied"
        ),

    "apostrophe_standardisation_applied":
        normalisation_integrity.get(
            "apostrophe_standardisation_applied"
        ),

    "dash_and_minus_standardisation_applied":
        normalisation_integrity.get(
            "dash_and_minus_standardisation_applied"
        ),

    "soft_hyphen_removal_applied":
        normalisation_integrity.get(
            "soft_hyphen_removal_applied"
        ),

    "semantic_harmonisation_applied":
        normalisation_integrity.get(
            "semantic_harmonisation_applied"
        ),

    "semantic_rewriting_applied":
        normalisation_integrity.get(
            "semantic_rewriting_applied"
        ),

    "unit_conversion_applied":
        normalisation_integrity.get(
            "unit_conversion_applied"
        ),

    "numeric_calculation_applied":
        normalisation_integrity.get(
            "numeric_calculation_applied"
        ),

    "manual_correction_applied":
        normalisation_integrity.get(
            "manual_correction_applied"
        ),

    "reference_values_used_for_transformation":
        normalisation_integrity.get(
            "reference_values_used_for_transformation"
        ),

    "numeric_token_preservation":
        normalisation_integrity.get(
            "numeric_token_preservation"
        ),

    "component_checks":
        normalisation_integrity.get(
            "component_checks"
        ),

    "representative_content_checks":
        normalisation_integrity.get(
            "representative_content_checks"
        )
}

print("Branch C representation integrity:")
print(json.dumps(
    representation_integrity,
    indent=2,
    ensure_ascii=False
))

Branch C representation integrity:
{
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "page_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "all_expected_components_preserved": true,
  "all_representative_content_preserved": true,
  "numeric_values_preserved": true,
  "all_scope_pages_present": true,
  "complete_12_page_representation_retained": true,
  "source_scope_filtering_applied": false,
  "page_removal_applied": false,
  "page_cropping_applied": false,
  "ocr_applied": false,
  "unicode_nfkc_normalisation_applied": true,
  "unicode_space_standardisation_applied": true,
  "apostrophe_standardisation_applied": true,
  "dash_and_minus_standardisation_applied": true,
  "soft_hyphen_removal_applied": true,
  "semantic_harmonisation_applied": false,
  "semantic_rewriting_applied": false,
  "unit_conversion_applied": false,
  "numeric_calculation_applied": false,
  "manual_correction_applied": false,


In [54]:
# ============================================================
# 25. Create reproducible Branch C validation metrics
# ============================================================

field_accuracy_dictionary = {
    row["Field"]: (
        None
        if pd.isna(row["Accuracy"])
        else float(row["Accuracy"])
    )
    for _, row in field_validation_df.iterrows()
}

category_metrics_dictionary = {
    row["Category"]: {
        "expected_records":
            int(row["Expected Records"]),
        "extracted_records":
            int(row["Extracted Records"]),
        "aligned_records":
            int(row["Aligned Records"]),
        "fully_correct_records":
            int(row["Fully Correct Records"]),
        "discrepant_records":
            int(row["Discrepant Records"]),
        "completeness":
            (
                None
                if pd.isna(row["Completeness"])
                else float(row["Completeness"])
            ),
        "record_precision_exact":
            float(row["Record Precision Exact"]),
        "record_recall_exact":
            float(row["Record Recall Exact"]),
        "record_f1_exact":
            float(row["Record F1 Exact"])
    }
    for _, row in category_metrics_df.iterrows()
}

equivalence_rule_usage = {
    "metric":
        int(comparison_df["Metric Equivalence Rule Applied"].sum())
        if not comparison_df.empty else 0,
    "topic":
        int(comparison_df["Topic Equivalence Rule Applied"].sum())
        if not comparison_df.empty else 0,
    "statement_or_section":
        int(comparison_df["Statement Equivalence Rule Applied"].sum())
        if not comparison_df.empty else 0,
    "unit":
        int(comparison_df["Unit Equivalence Rule Applied"].sum())
        if not comparison_df.empty else 0,
    "qualifier":
        int(comparison_df["Qualifier Equivalence Rule Applied"].sum())
        if not comparison_df.empty else 0,
    "reporting_period":
        int(comparison_df["Reporting Period Equivalence Rule Applied"].sum())
        if not comparison_df.empty else 0
}

VALIDATION_METRICS = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "input_representation": INPUT_REPRESENTATION,

    "reference_records": int(reference_record_count),
    "extracted_records": int(extracted_record_count),
    "aligned_records": int(aligned_record_count),
    "fully_correct_records": int(fully_correct_record_count),
    "discrepant_records": int(discrepant_record_count),
    "missing_records": int(missing_record_count),
    "unsupported_extracted_records": int(unsupported_record_count),

    "completeness": round(float(completeness), 4),
    "missing_rate": round(float(missing_rate), 4),
    "record_precision_exact": round(float(record_precision_exact), 4),
    "record_recall_exact": round(float(record_recall_exact), 4),
    "record_f1_exact": round(float(record_f1_exact), 4),
    "unsupported_rate": round(float(unsupported_rate), 4),
    "discrepancy_rate_among_aligned":
        round(float(discrepancy_rate_among_aligned), 4),

    "overall_primary_field_accuracy":
        (
            None
            if overall_primary_field_accuracy is None
            else round(float(overall_primary_field_accuracy), 4)
        ),

    "sign_only_value_difference_count":
        int(sign_only_value_difference_count),

    "field_accuracy_among_aligned":
        field_accuracy_dictionary,

    "schema_validity": bool(schema_validity),
    "schema_diagnostics": schema_diagnostics,
    "content_diagnostics": content_diagnostics,

    "branch_C_representation_integrity":
        representation_integrity,

    "matching_rules": {
        "blocking_fields": BLOCK_FIELDS,
        "one_to_one_assignment":
            "Hungarian linear-sum assignment",
        "matching_score_threshold":
            MATCH_SCORE_THRESHOLD,
        "matching_score_weights":
            MATCHING_WEIGHTS,
        "value_used_for_alignment": False,
        "unit_used_for_alignment": False,
        "qualifier_used_for_alignment": False,
        "controlled_qualitative_equivalence_used_for_identity": True
    },

    "comparison_rules": {
        "raw_extraction_modified": False,
        "manual_correction_applied": False,
        "unexpected_fields_copied_to_expected_fields": False,
        "comparison_normalisation_scope":
            "Comparison copies only",
        "null_comparison":
            "None and pandas NaN treated as equivalent absence",
        "numeric_comparison":
            "Signed numeric equality after deterministic parsing",
        "absolute_numeric_value_for_correctness": False,
        "absolute_numeric_value_for_alignment": False,
        "sign_only_difference":
            (
                "Diagnostic only; signed Value remains "
                "authoritative for correctness"
            ),
        "statement_or_section":
            (
                "Physical paragraph/recommendation identity plus "
                "frozen source-grounded D7 statement/section alternatives"
            ),
        "metric":
            (
                "Normalised exact equality or frozen source-grounded "
                "D7 equivalence pair; lexical similarity is diagnostic only"
            ),
        "topic":
            (
                "Normalised exact equality or frozen source-grounded "
                "D7 equivalence pair; lexical similarity is diagnostic only"
            ),
        "unit":
            (
                "Controlled notation and source-grounded unit equivalence only; "
                "qualifier wording is never moved into Unit"
            ),
        "qualifier":
            (
                "Normalised exact equality plus the source-grounded "
                "minimum / and upwards equivalence; emphasis words such as "
                "only, just and additional are not treated as null-equivalent"
            ),
        "reporting_period":
            "Frozen source-grounded D7 period canonicalisation",
        "source_location":
            "Normalised exact correctness",
        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,
        "d7_equivalence_rules_status":
            (
                "Frozen D7 document/schema-level source-grounded "
                "equivalence rules; reused unchanged from Branch A"
            )
    },

    "equivalence_rule_usage":
        equivalence_rule_usage,

    "category_metrics":
        category_metrics_dictionary,

    "normalisation_note":
        (
            "Deterministic normalisation and frozen source-grounded "
            "equivalence rules were applied only during comparison; "
            "the preserved Branch C extraction was not modified."
        ),

    "input_provenance": {
        "reference_file":
            REFERENCE_PATH.name,
        "reference_sha256":
            REFERENCE_SHA256,
        "parsed_extraction_file":
            EXTRACTION_PATH.name,
        "parsed_extraction_sha256":
            EXTRACTION_SHA256,
        "structure_check_file":
            STRUCTURE_CHECK_PATH.name,
        "structure_check_sha256":
            STRUCTURE_CHECK_SHA256,
        "experiment_metadata_file":
            EXPERIMENT_METADATA_PATH.name,
        "experiment_metadata_sha256":
            EXPERIMENT_METADATA_SHA256,
        "normalisation_integrity_file":
            NORMALISATION_INTEGRITY_PATH.name,
        "normalisation_integrity_sha256":
            NORMALISATION_INTEGRITY_SHA256,
        "experiment_summary_file":
            (
                EXPERIMENT_SUMMARY_PATH.name
                if EXPERIMENT_SUMMARY_PATH is not None
                else None
            ),
        "experiment_summary_sha256":
            EXPERIMENT_SUMMARY_SHA256,
        "branch_C_structure_valid":
            bool(branch_c_structure_valid),
        "parsed_extraction_hash_matches_metadata":
            bool(parsed_extraction_hash_matches_metadata),
        "source_hash_matches_stage_1":
            bool(metadata_source_hash_correct)
    },

    "comparison_rules_frozen_from_branch_A": True,
    "validation_timestamp":
        datetime.now().isoformat()
}

print(
    json.dumps(
        VALIDATION_METRICS,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D7",
  "document_name": "UK National Audit Office — Delivering STEM (science, technology, engineering and mathematics) skills for the economy",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised structural Markdown",
  "reference_records": 59,
  "extracted_records": 60,
  "aligned_records": 59,
  "fully_correct_records": 11,
  "discrepant_records": 48,
  "missing_records": 0,
  "unsupported_extracted_records": 1,
  "completeness": 1.0,
  "missing_rate": 0.0,
  "record_precision_exact": 0.1833,
  "record_recall_exact": 0.1864,
  "record_f1_exact": 0.1849,
  "unsupported_rate": 0.0167,
  "discrepancy_rate_among_aligned": 0.8136,
  "overall_primary_field_accuracy": 0.855,
  "sign_only_value_difference_count": 0,
  "field_accuracy_among_aligned": {
    "Category": 1.0,
    "Statement or Section": 0.5084745762711864,
    "Metric": 0.6271186440677966,
    "Topic": 0.6779661016949152,
    "Value

In [55]:
# ============================================================
# 26. Validation metadata and conclusion
# ============================================================

VALIDATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,

    "reference_file":
        REFERENCE_PATH.name,

    "reference_file_sha256":
        REFERENCE_SHA256,

    "extraction_file":
        EXTRACTION_PATH.name,

    "extraction_file_sha256":
        EXTRACTION_SHA256,

    "structure_check_file":
        STRUCTURE_CHECK_PATH.name,

    "structure_check_file_sha256":
        STRUCTURE_CHECK_SHA256,

    "experiment_metadata_file":
        EXPERIMENT_METADATA_PATH.name,

    "experiment_metadata_file_sha256":
        EXPERIMENT_METADATA_SHA256,

    "normalisation_integrity_file":
        NORMALISATION_INTEGRITY_PATH.name,

    "normalisation_integrity_file_sha256":
        NORMALISATION_INTEGRITY_SHA256,

    "validation_type":
        (
            "Deterministic comparison against the fixed "
            "59-record D7 Stage 1 reference dataset"
        ),

    "raw_extraction_modified": False,
    "manual_correction_applied": False,
    "schema_errors_preserved": True,

    "comparison_normalisation_scope":
        "Comparison copies only",

    "matching_outcome_values_used": False,

    "automatic_unmatched_label":
        "unsupported/unmatched; not automatically hallucinated",

    "comparison_rules_frozen_from_branch_A": True,

    "created_at":
        datetime.now().isoformat(),

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "notes":
        (
            "Validation C compares the fixed D7 Stage 1 reference "
            "dataset with the exact canonical Branch C parsed extraction. "
            "All nine Stage 1 fields, including Qualifier, are evaluated. "
            "Value, Unit and Qualifier are excluded from alignment. "
            "The final D7 Branch A source-grounded equivalence rules are "
            "reused unchanged."
        )
}


validation_status = (
    "Completed without discrepancies"
    if (
        fully_correct_record_count
        == reference_record_count
        and missing_record_count == 0
        and unsupported_record_count == 0
        and schema_validity
    )
    else "Completed with discrepancies"
)


VALIDATION_CONCLUSION = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,

    "validation_status":
        validation_status,

    "reference_records":
        int(reference_record_count),

    "extracted_records":
        int(extracted_record_count),

    "aligned_records":
        int(aligned_record_count),

    "fully_correct_records":
        int(fully_correct_record_count),

    "discrepant_records":
        int(discrepant_record_count),

    "missing_records":
        int(missing_record_count),

    "unsupported_extracted_records":
        int(unsupported_record_count),

    "completeness":
        round(float(completeness), 4),

    "record_precision_exact":
        round(float(record_precision_exact), 4),

    "record_recall_exact":
        round(float(record_recall_exact), 4),

    "record_f1_exact":
        round(float(record_f1_exact), 4),

    "overall_primary_field_accuracy":
        (
            None
            if overall_primary_field_accuracy is None
            else round(float(overall_primary_field_accuracy), 4)
        ),

    "sign_only_value_difference_count":
        int(sign_only_value_difference_count),

    "schema_valid":
        bool(schema_validity),

    "normalisation_integrity_passed":
        representation_integrity[
            "normalisation_integrity_passed"
        ],

    "notes":
        (
            "Schema validity, Branch C normalisation integrity, "
            "completeness, record correspondence and field-level "
            "correctness are reported as separate outcomes."
        )
}

print(json.dumps(
    VALIDATION_CONCLUSION,
    ensure_ascii=False,
    indent=2
))

{
  "document_id": "D7",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "validation_status": "Completed with discrepancies",
  "reference_records": 59,
  "extracted_records": 60,
  "aligned_records": 59,
  "fully_correct_records": 11,
  "discrepant_records": 48,
  "missing_records": 0,
  "unsupported_extracted_records": 1,
  "completeness": 1.0,
  "record_precision_exact": 0.1833,
  "record_recall_exact": 0.1864,
  "record_f1_exact": 0.1849,
  "overall_primary_field_accuracy": 0.855,
  "sign_only_value_difference_count": 0,
  "schema_valid": true,
  "normalisation_integrity_passed": true,
  "notes": "Schema validity, Branch C normalisation integrity, completeness, record correspondence and field-level correctness are reported as separate outcomes."
}


In [56]:
# ============================================================
# 27. Define and export Validation C outputs
# ============================================================

DETAILED_PATH = (
    OUTPUT_DIR / "D7_branch_C_validation_detailed.csv"
)

FULLY_CORRECT_RECORDS_PATH = (
    OUTPUT_DIR / "D7_branch_C_fully_correct_records.csv"
)

DISCREPANT_RECORDS_PATH = (
    OUTPUT_DIR / "D7_branch_C_discrepant_records.csv"
)

MISSING_RECORDS_PATH = (
    OUTPUT_DIR / "D7_branch_C_missing_records.csv"
)

UNSUPPORTED_RECORDS_PATH = (
    OUTPUT_DIR / "D7_branch_C_unsupported_records.csv"
)

SCHEMA_ISSUES_PATH = (
    OUTPUT_DIR / "D7_branch_C_schema_issues.csv"
)

TYPE_ISSUES_PATH = (
    OUTPUT_DIR / "D7_branch_C_type_issues.csv"
)

FIELD_VALIDATION_PATH = (
    OUTPUT_DIR / "D7_branch_C_field_validation.csv"
)

FIELD_ERROR_SUMMARY_PATH = (
    OUTPUT_DIR / "D7_branch_C_field_error_summary.csv"
)

CATEGORY_METRICS_PATH = (
    OUTPUT_DIR / "D7_branch_C_category_metrics.csv"
)

VALIDATION_SUMMARY_CSV_PATH = (
    OUTPUT_DIR / "D7_branch_C_validation_summary.csv"
)

VALIDATION_SUMMARY_JSON_PATH = (
    OUTPUT_DIR / "D7_branch_C_validation_summary.json"
)

VALIDATION_METADATA_PATH = (
    OUTPUT_DIR / "D7_branch_C_validation_metadata.json"
)

VALIDATION_CONCLUSION_PATH = (
    OUTPUT_DIR / "D7_branch_C_validation_conclusion.json"
)


comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig"
)

fully_correct_records_df.to_csv(
    FULLY_CORRECT_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

discrepant_records_df.to_csv(
    DISCREPANT_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

missing_records_df.to_csv(
    MISSING_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

unsupported_records_df.to_csv(
    UNSUPPORTED_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

schema_issues_df.to_csv(
    SCHEMA_ISSUES_PATH,
    index=False,
    encoding="utf-8-sig"
)

type_issues_df.to_csv(
    TYPE_ISSUES_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_validation_df.to_csv(
    FIELD_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_error_summary_df.to_csv(
    FIELD_ERROR_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

category_metrics_df.to_csv(
    CATEGORY_METRICS_PATH,
    index=False,
    encoding="utf-8-sig"
)

validation_summary_df.to_csv(
    VALIDATION_SUMMARY_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

VALIDATION_SUMMARY_JSON_PATH.write_text(
    json.dumps(
        VALIDATION_METRICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

VALIDATION_METADATA_PATH.write_text(
    json.dumps(
        VALIDATION_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

VALIDATION_CONCLUSION_PATH.write_text(
    json.dumps(
        VALIDATION_CONCLUSION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("Validation artefacts saved.")

Validation artefacts saved.


In [57]:
# ============================================================
# 28. Final validation consistency checks
# ============================================================

if not reference_schema_valid:
    raise AssertionError(
        "Reference schema validation failed."
    )

if not reference_record_count_valid:
    raise AssertionError(
        "Reference record-count validation failed."
    )

if not reference_category_counts_valid:
    raise AssertionError(
        "Reference category-count validation failed."
    )

if not field_types_valid:
    raise AssertionError(
        "The extracted comparison fields contain invalid data types."
    )

if not representation_integrity[
    "normalisation_integrity_passed"
]:
    raise AssertionError(
        "Branch C normalisation-integrity checks did not pass."
    )

if (
    aligned_record_count
    + missing_record_count
    != reference_record_count
):
    raise AssertionError(
        "Reference-record accounting is inconsistent."
    )

if (
    aligned_record_count
    + unsupported_record_count
    != extracted_record_count
):
    raise AssertionError(
        "Extraction-record accounting is inconsistent."
    )

if (
    fully_correct_record_count
    + discrepant_record_count
    != aligned_record_count
):
    raise AssertionError(
        "Aligned-record correctness accounting is inconsistent."
    )

print("Validation status:", validation_status)
print("Reference records:", reference_record_count)
print("Extracted records:", extracted_record_count)
print("Aligned records:", aligned_record_count)
print("Missing records:", missing_record_count)
print("Unsupported/unmatched records:", unsupported_record_count)
print("Fully correct records:", fully_correct_record_count)
print("Discrepant records:", discrepant_record_count)
print("Schema valid:", schema_validity)
print(
    "Normalisation integrity passed:",
    representation_integrity[
        "normalisation_integrity_passed"
    ]
)
print(
    "Sign-only value differences:",
    sign_only_value_difference_count
)
print("Exact F1:", record_f1_exact)
print(
    "Overall primary field accuracy:",
    overall_primary_field_accuracy
)

print("\nD7 Validation C completed successfully.")

Validation status: Completed with discrepancies
Reference records: 59
Extracted records: 60
Aligned records: 59
Missing records: 0
Unsupported/unmatched records: 1
Fully correct records: 11
Discrepant records: 48
Schema valid: True
Normalisation integrity passed: True
Sign-only value differences: 0
Exact F1: 0.18487394957983194
Overall primary field accuracy: 0.8549905838041432

D7 Validation C completed successfully.


In [58]:
# ============================================================
# 29. Download generated Validation C outputs
# ============================================================

GENERATED_OUTPUTS = [
    DETAILED_PATH,
    FULLY_CORRECT_RECORDS_PATH,
    DISCREPANT_RECORDS_PATH,
    MISSING_RECORDS_PATH,
    UNSUPPORTED_RECORDS_PATH,
    SCHEMA_ISSUES_PATH,
    TYPE_ISSUES_PATH,
    FIELD_VALIDATION_PATH,
    FIELD_ERROR_SUMMARY_PATH,
    CATEGORY_METRICS_PATH,
    VALIDATION_SUMMARY_CSV_PATH,
    VALIDATION_SUMMARY_JSON_PATH,
    VALIDATION_METADATA_PATH,
    VALIDATION_CONCLUSION_PATH
]

print("Generated D7 Validation C files:\n")

for output_path in GENERATED_OUTPUTS:
    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists()
    )

for output_path in GENERATED_OUTPUTS:
    if output_path.exists():
        files.download(output_path)

Generated D7 Validation C files:

- D7_branch_C_validation_detailed.csv | exists: True
- D7_branch_C_fully_correct_records.csv | exists: True
- D7_branch_C_discrepant_records.csv | exists: True
- D7_branch_C_missing_records.csv | exists: True
- D7_branch_C_unsupported_records.csv | exists: True
- D7_branch_C_schema_issues.csv | exists: True
- D7_branch_C_type_issues.csv | exists: True
- D7_branch_C_field_validation.csv | exists: True
- D7_branch_C_field_error_summary.csv | exists: True
- D7_branch_C_category_metrics.csv | exists: True
- D7_branch_C_validation_summary.csv | exists: True
- D7_branch_C_validation_summary.json | exists: True
- D7_branch_C_validation_metadata.json | exists: True
- D7_branch_C_validation_conclusion.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>